# PCVRHyFormer 模型全量走读 Notebook

> **目标**：把 baseline `train/model.py`（1714 行 / 16 个类）全部搬进 notebook，逐段中文讲解，**每行吃透**。
> 数据 / 训练循环仍直接复用 baseline 的 `train/dataset.py` 与 `train/trainer.py`，**不改一行 baseline 源码**。

## 阅读路径
| 章节 | 内容 | 涉及 baseline 行号 |
|---|---|---|
| 0 | 环境与依赖 | — |
| 1 | `ModelInput`：模型 forward 的统一输入容器 | 1–18 |
| 2 | `RotaryEmbedding`：RoPE 位置编码缓存 | 26–64 |
| 3 | `rotate_half` / `apply_rope_to_tensor`：旋转工具 | 67–92 |
| 4 | `SwiGLU`：GLU 门控激活 | 100–114 |
| 5 | `RoPEMultiheadAttention`：带 RoPE 的多头注意力 + 输出门控 G | 117–241 |
| 6 | `CrossAttention`：Pre-LN 交叉注意力封装 | 244–312 |
| 7 | `RankMixerBlock`：Token Mixing + per-token FFN | 315–412 |
| 8 | `MultiSeqQueryGenerator`：每个序列独立生成 Q tokens | 415–494 |
| 9 | `SwiGLUEncoder`：无注意力序列编码器 | 502–541 |
| 10 | `TransformerEncoder`：标准 Transformer 编码层 | 544–614 |
| 11 | `LongerEncoder`：top-K 压缩编码器（cross/self 自适应） | 616–808 |
| 12 | `create_sequence_encoder`：编码器工厂函数 | 811–842 |
| 13 | `MultiSeqHyFormerBlock`：多序列 HyFormer 主块 | 850–980 |
| 14 | `GroupNSTokenizer`：按 group 一格一 token | 988–1067 |
| 15 | `RankMixerNSTokenizer`：cat→split→project | 1070–1189 |
| 16 | `PCVRHyFormer`：主模型 init / 私有方法 / forward / predict | 1192–1714 |
| 17 | 数据准备（沿用上一版 LFS 兜底逻辑） | — |
| 18 | 实例化模型并做单 batch 烟囱测试 | — |
| 19 | 调用 baseline trainer 训练 | — |
| 20 | 评估 + 单 batch 推理 | — |

> 约定：每个章节先给出 `# === baseline 行 X-Y ===` 的代码 cell（**与 model.py 完全一致**），再用 markdown 解释每段做什么、为何这样写。


## Step 0 · 环境与依赖

把 `train/` 加入 `sys.path`。**故意不 import baseline 的 `model.py`**——我们要在 notebook 里自己重新定义所有模型类。
后面只会从 baseline 复用 [`get_pcvr_data`](train/dataset.py)、[`PCVRHyFormerRankingTrainer`](train/trainer.py) 等「与模型解耦」的工具。


In [1]:
import sys, os, math, json, logging, random, warnings, glob
from pathlib import Path
from typing import List, NamedTuple, Tuple, Optional, Union, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

# 项目路径（兼容 cwd=项目根 或 cwd=taac-codes 两种情况）
PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'taac-codes').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'taac-codes'

TRAIN_DIR  = PROJECT_ROOT / 'train'
SAMPLE_DIR = PROJECT_ROOT / 'sample_data'
WORK_DIR   = PROJECT_ROOT / 'playground_workdir'
WORK_DIR.mkdir(exist_ok=True)

assert TRAIN_DIR.exists(), f'Cannot find {TRAIN_DIR}'
if str(TRAIN_DIR) not in sys.path:
    sys.path.insert(0, str(TRAIN_DIR))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch     :', torch.__version__, '| device:', DEVICE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('TRAIN_DIR   :', TRAIN_DIR)
print('WORK_DIR    :', WORK_DIR)


PyTorch     : 2.11.0 | device: cpu
PROJECT_ROOT: /Users/huangjing103/PyCharmMiscProject/taac-codes
TRAIN_DIR   : /Users/huangjing103/PyCharmMiscProject/taac-codes/train
WORK_DIR    : /Users/huangjing103/PyCharmMiscProject/taac-codes/playground_workdir


## Section 1 · `ModelInput` —— 模型输入容器

PCVRHyFormer 的 `forward` 只接一个参数 `inputs: ModelInput`，里面把所有特征打包成 NamedTuple。
NamedTuple 既能像 tuple 那样解包，也能用属性 `inputs.user_int_feats` 访问，等价于 dataclass 但更轻量。

字段含义：
- `user_int_feats`/`item_int_feats`: `(B, total_int_dim)`，所有 user/item 离散特征 concat 后的 long 张量；NS Tokenizer 会按 `feature_specs` 切片去查 Embedding。
- `user_dense_feats`/`item_dense_feats`: `(B, dense_dim)`，连续特征，单独走 MLP 投到 d_model 形成 1 个 NS token。
- `seq_data`: `dict[domain, (B, S_sideinfo, L)]`，每个 domain 的序列；最里层 `S_sideinfo` 是同一时间步上不同 fid 的取值（譬如 photo_id / channel_id / cate_id…），`L` 是序列长度。
- `seq_lens`: `dict[domain, (B,)]`，每条序列的真实有效长度。
- `seq_time_buckets`: `dict[domain, (B, L)]`，每个时间位的「时间差桶号」（0~64），喂给 `time_embedding`。


In [2]:
# === baseline train/model.py 第 11-18 行 ===
class ModelInput(NamedTuple):
    user_int_feats: torch.Tensor
    item_int_feats: torch.Tensor
    user_dense_feats: torch.Tensor
    item_dense_feats: torch.Tensor
    seq_data: dict        # {domain: tensor [B, S, L]}
    seq_lens: dict        # {domain: tensor [B]}
    seq_time_buckets: dict  # {domain: tensor [B, L]}


## Section 2 · `RotaryEmbedding` —— RoPE 位置编码缓存

RoPE（Rotary Position Embedding）核心思想：把绝对位置 `p` 编码成「在每对维度上旋转一个角度 `p·θ_k`」，然后让点积 `Q·K` 自动变成「相对位置」依赖。

逐行拆解：
- `inv_freq = 1 / (base^(2k/dim))`：传统 sinusoidal 同款的频率倒数，`dim/2` 个。
- `register_buffer(persistent=False)`：作为 buffer 跟着 `model.to(device)` 走，但不进 state_dict（节省存储）。
- `_build_cache(seq_len)`：用 `torch.outer(t, inv_freq)` 得到 `(seq_len, dim/2)` 的相位矩阵，然后 `cat([freqs, freqs], -1)` 复制成 `(seq_len, dim)`，再分别取 `cos`/`sin` 缓存成 `(1, seq_len, dim)`，便于后续直接乘 `(B, h, L, head_dim)`。
- `forward(seq_len, device)`：直接切片返回，**不做运行时扩容**——这是为了和 `torch.compile` 兼容（`torch.compile` 不喜欢动态形状）。


In [3]:
# === baseline train/model.py 第 26-64 行 ===
class RotaryEmbedding(nn.Module):
    """Precomputes and caches RoPE cos/sin values.

    Attributes:
        dim: Rotary embedding dimension.
        max_seq_len: Maximum sequence length for cache.
        base: Base frequency for rotary encoding.
    """

    def __init__(self, dim: int, max_seq_len: int = 2048, base: float = 10000.0) -> None:
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.base = base

        # Precompute inv_freq: (dim // 2,)
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)

        # Precompute cache
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len: int) -> None:
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype, device=self.inv_freq.device)
        freqs = torch.outer(t, self.inv_freq)  # (seq_len, dim // 2)
        emb = torch.cat([freqs, freqs], dim=-1)  # (seq_len, dim)
        self.register_buffer('cos_cached', emb.cos().unsqueeze(0), persistent=False)  # (1, seq_len, dim)
        self.register_buffer('sin_cached', emb.sin().unsqueeze(0), persistent=False)  # (1, seq_len, dim)

    def forward(self, seq_len: int, device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
        """Computes cos/sin values for the given sequence length.

        Returns pre-computed slices from the cache. The cache is built once
        in __init__ with max_seq_len; no runtime expansion is performed so
        that the forward pass remains compatible with torch.compile().
        """
        cos = self.cos_cached[:, :seq_len, :].to(device)
        sin = self.sin_cached[:, :seq_len, :].to(device)
        return cos, sin


## Section 3 · `rotate_half` + `apply_rope_to_tensor` —— 旋转工具

旋转矩阵在数值上等价于：`x ⊙ cos + rotate_half(x) ⊙ sin`，其中 `rotate_half([a, b]) = [-b, a]`（这里是块状版本：把后一半取负、放到前面）。

- `rotate_half`：`x` 最后一维 `[..., -2*half:]` 取负，与前一半 swap。这是 RoPE 论文里的关键代数恒等式：让 `Q@K^T` 自动产生相对位置因子。
- `apply_rope_to_tensor`：把 `(1, L_max, head_dim)` 的 `cos/sin` 切到当前 `L`，再 `unsqueeze(1)` 补 head 维度，广播乘 `(B, num_heads, L, head_dim)` 的 `Q`/`K`。
- 注意：cos/sin 也接受 `(B, L, head_dim)`，这就是 `LongerEncoder` 在 cross-attn 时为「按位置 gather 出来的 top_k 位置」单独传 RoPE 的入口。


In [4]:
# === baseline train/model.py 第 67-92 行 ===
def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Swaps and negates the first and second halves of the last dimension."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope_to_tensor(
    x: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> torch.Tensor:
    """Applies Rotary Position Embedding to a single tensor.

    Args:
        x: (B, num_heads, L, head_dim)
        cos: (1, L_max, head_dim) or (B, L, head_dim) for batch-specific positions.
        sin: Same shape as cos.

    Returns:
        Rotated tensor of shape (B, num_heads, L, head_dim).
    """
    L = x.shape[2]
    cos_ = cos[:, :L, :].unsqueeze(1)  # (*, 1, L, head_dim)
    sin_ = sin[:, :L, :].unsqueeze(1)
    return x * cos_ + rotate_half(x) * sin_


## Section 4 · `SwiGLU` —— 门控激活

SwiGLU = Swish-Gated Linear Unit，PaLM/LLaMA 中常用的激活，比 GELU 表达力更强。

- `fc(d_model → 2*hidden)`：一次性出两条支路 `x1`、`x2`。
- `x1 * silu(x2)`：`x1` 是「值」，`silu(x2)` 是「门」；门控让信号能选择性通过。
- `fc_out(hidden → d_model)`：投影回 `d_model` 维度。
- `hidden_mult=4`：FFN 隐层倍数，与 vanilla Transformer 的 4× 一致。


In [5]:
# === baseline train/model.py 第 100-114 行 ===
class SwiGLU(nn.Module):
    """SwiGLU activation: x1 * SiLU(x2)."""

    def __init__(self, d_model: int, hidden_mult: int = 4) -> None:
        super().__init__()
        hidden_dim = d_model * hidden_mult
        self.fc = nn.Linear(d_model, 2 * hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc(x)
        x1, x2 = x.chunk(2, dim=-1)
        x = x1 * F.silu(x2)
        x = self.fc_out(x)
        return x


## Section 5 · `RoPEMultiheadAttention` —— 带 RoPE 的多头注意力

整个模型最复杂的一个原子算子。下面按 forward 的步骤逐段拆解：

### 5.1 init 关键设计
- `W_q/W_k/W_v/W_o`：标准 4 个线性投影。
- `W_g`：**输出门控**，权重置 0、bias 置 1.0 → 训练初期 `sigmoid(G)≈sigmoid(1)≈0.73`，相当于「温和地做了 73% 的注意力 + 27% 的零」。这是工程稳定性 trick：让模型从近似恒等开始学。
- `rope_on_q`：CrossAttention 里设为 False（query 是 NS/Q token，不带位置），普通 self-attention 设为 True。

### 5.2 forward 6 步
1. **Linear projection**：`Q/K/V = W_*(query/key/value)`；shape `(B, L, D)`。
2. **Reshape to multi-head**：`(B, L, num_heads, head_dim)` 然后 `transpose(1, 2)` 变 `(B, num_heads, L, head_dim)`。
3. **Apply RoPE**：K 永远 apply（KV 侧位置）；Q 看 `rope_on_q`，并且优先用 `q_rope_cos/sin`（这是 `LongerEncoder` 在 top-K cross-attn 时为不连续位置单独算的 RoPE）。
4. **Mask 转换**：`key_padding_mask: (B, Lk) bool, True=pad` → SDPA 要的 `(B, num_heads, Lq, Lk) bool, True=attend`，所以取反 + 扩展。`attn_mask`（如 causal）的合并也在这一步。
5. **SDPA**：`F.scaled_dot_product_attention` 自动选最优后端（FlashAttention/Memory-Efficient/数学版）。`nan_to_num(out, nan=0.0)`：当某 query 对应的 K 全是 padding，softmax 全 0 会出 NaN，这里强制置零，配合残差就等于「保持原 query」。
6. **输出投影**：reshape 回 `(B, L, D)` → 乘 `sigmoid(W_g(query))` 做输出门控 → `W_o`。


In [6]:
# === baseline train/model.py 第 117-241 行 ===
class RoPEMultiheadAttention(nn.Module):
    """Multi-head attention with Rotary Position Embedding support.

    Manually projects Q/K/V and reshapes for multi-head, then injects RoPE
    after projection and before dot-product. Uses F.scaled_dot_product_attention
    for efficient computation.
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        dropout: float = 0.0,
        rope_on_q: bool = True,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.rope_on_q = rope_on_q
        self.dropout = dropout

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.W_g = nn.Linear(d_model, d_model)

        nn.init.zeros_(self.W_g.weight)
        nn.init.constant_(self.W_g.bias, 1.0)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        key_padding_mask: Optional[torch.Tensor] = None,
        attn_mask: Optional[torch.Tensor] = None,
        rope_cos: Optional[torch.Tensor] = None,
        rope_sin: Optional[torch.Tensor] = None,
        q_rope_cos: Optional[torch.Tensor] = None,
        q_rope_sin: Optional[torch.Tensor] = None,
        need_weights: bool = False,
    ) -> tuple:
        """Computes multi-head attention with optional RoPE.

        Args:
            query: (B, Lq, D)
            key: (B, Lk, D)
            value: (B, Lk, D)
            key_padding_mask: (B, Lk), True indicates padding positions.
            attn_mask: (Lq, Lk) or (B*num_heads, Lq, Lk), additive mask.
            rope_cos: (1, L, head_dim), RoPE for KV side (also used for Q
                unless q_rope_* is provided).
            rope_sin: Same shape as rope_cos.
            q_rope_cos: (B, Lq, head_dim) or (1, Lq, head_dim), Q-specific
                RoPE for cross-attention with gathered positions.
            q_rope_sin: Same shape as q_rope_cos.
            need_weights: Compatibility parameter, not used.

        Returns:
            Tuple of (output, None).
        """
        B, Lq, _ = query.shape
        Lk = key.shape[1]

        # 1. Linear projection
        Q = self.W_q(query)  # (B, Lq, D)
        K = self.W_k(key)    # (B, Lk, D)
        V = self.W_v(value)  # (B, Lk, D)

        # 2. Reshape to (B, num_heads, L, head_dim)
        Q = Q.view(B, Lq, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, Lk, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, Lk, self.num_heads, self.head_dim).transpose(1, 2)

        # 3. Apply RoPE independently to Q and K
        if rope_cos is not None and rope_sin is not None:
            # K always uses rope_cos/rope_sin (KV-side positional encoding)
            K = apply_rope_to_tensor(K, rope_cos, rope_sin)

            if self.rope_on_q:
                # Q side: prefer dedicated q_rope_cos/sin (top_k positions in LongerEncoder cross-attn)
                q_cos = q_rope_cos if q_rope_cos is not None else rope_cos
                q_sin = q_rope_sin if q_rope_sin is not None else rope_sin
                Q = apply_rope_to_tensor(Q, q_cos, q_sin)

        # 4. Convert key_padding_mask to SDPA format
        sdpa_attn_mask = None
        if key_padding_mask is not None:
            # key_padding_mask: (B, Lk), True = padding
            # SDPA expects (B, 1, 1, Lk) bool mask, True = attend
            sdpa_attn_mask = ~key_padding_mask.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, Lk)
            sdpa_attn_mask = sdpa_attn_mask.expand(B, self.num_heads, Lq, Lk)

        if attn_mask is not None:
            # attn_mask: additive float mask (Lq, Lk), -inf means do not attend
            # Convert to bool: positions that are not -inf are True
            bool_attn = (attn_mask == 0)  # (Lq, Lk)
            bool_attn = bool_attn.unsqueeze(0).unsqueeze(0).expand(B, self.num_heads, Lq, Lk)
            if sdpa_attn_mask is not None:
                sdpa_attn_mask = sdpa_attn_mask & bool_attn
            else:
                sdpa_attn_mask = bool_attn

        # 5. Scaled Dot-Product Attention
        dropout_p = self.dropout if self.training else 0.0
        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=sdpa_attn_mask,
            dropout_p=dropout_p,
        )  # (B, num_heads, Lq, head_dim)

        # Replace NaN from all-padding softmax with 0 (zero vectors preserve original input via residual)
        out = torch.nan_to_num(out, nan=0.0)

        # 6. Reshape back and output projection
        out = out.transpose(1, 2).contiguous().view(B, Lq, self.d_model)
        G = self.W_g(query)
        out = out * torch.sigmoid(G)
        out = self.W_o(out)

        return out, None


## Section 6 · `CrossAttention` —— Pre-LN 交叉注意力封装

把 `RoPEMultiheadAttention` 包成「**Q 不带位置、KV 带 RoPE**」的 cross-attn 块，并加 LayerNorm。

- `rope_on_q=False`：Q 是 Q tokens（学出来的、无序的），不需要位置；K/V 是序列 token，需要位置。
- `ln_mode='pre'`（默认）：进 attention 之前对 Q 和 KV 各做一次 LayerNorm；这是 GPT/LLaMA 主流的 pre-norm 风格，比 post-norm 更稳定。
- `out = residual + out`：残差永远基于原始 query（未归一化）。
- `ln_mode='post'`：兼容老式 post-norm 路径，目前模型默认走 pre。


In [7]:
# === baseline train/model.py 第 244-312 行 ===
class CrossAttention(nn.Module):
    """Cross-attention module.

    Query comes from global tokens (Q tokens), Key/Value comes from sequence
    tokens. Only applies RoPE to KV side (rope_on_q=False).
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        dropout: float = 0.0,
        ln_mode: str = 'pre'
    ) -> None:
        super().__init__()
        self.ln_mode = ln_mode

        self.attn = RoPEMultiheadAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout,
            rope_on_q=False,
        )

        if ln_mode in ['pre', 'post']:
            self.norm_q = nn.LayerNorm(d_model)
            self.norm_kv = nn.LayerNorm(d_model)

    def forward(
        self,
        query: torch.Tensor,
        key_value: torch.Tensor,
        key_padding_mask: Optional[torch.Tensor] = None,
        rope_cos: Optional[torch.Tensor] = None,
        rope_sin: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Computes cross-attention between query tokens and sequence tokens.

        Args:
            query: (B, Nq, D), query tokens.
            key_value: (B, L, D), sequence tokens.
            key_padding_mask: (B, L), True indicates padding positions.
            rope_cos: (1, L, head_dim), KV-side RoPE cosine values.
            rope_sin: (1, L, head_dim), KV-side RoPE sine values.

        Returns:
            Output tensor of shape (B, Nq, D).
        """
        residual = query

        if self.ln_mode == 'pre':
            query = self.norm_q(query)
            key_value = self.norm_kv(key_value)

        out, _ = self.attn(
            query=query,
            key=key_value,
            value=key_value,
            key_padding_mask=key_padding_mask,
            rope_cos=rope_cos,
            rope_sin=rope_sin,
        )

        out = residual + out

        if self.ln_mode == 'post':
            out = self.norm_q(out)

        return out


## Section 7 · `RankMixerBlock` —— Token Mixing + per-token FFN

来自 RankMixer 论文（推荐排序专用 Transformer 变体）。要解决的问题：标准 Transformer 用 self-attention 实现 token 间交互、参数代价大；RankMixer 用 **零参数的 reshape** 完成 token mixing。

### 7.1 三种 mode
- `'full'`：先 token-mixing 再 FFN（约束 `d_model % T == 0`）。
- `'ffn_only'`：跳过 mixing，只有 per-token FFN。
- `'none'`：identity 直通。

### 7.2 token_mixing —— 参数量为 0 的「通道置换」
- 输入 `(B, T, D)`，把 `D` 切成 `T` 等份 `d_sub = D / T`，得到 `(B, T, T, d_sub)`。
- 现在 `(B, token, h, d_sub)`：`token` 是原 token 维，`h` 是子通道维。
- `transpose(1, 2)` 把 `token` 与 `h` 交换 → `(B, h, token, d_sub)`：每个原 token 的第 `k` 块子通道，被换到了第 `k` 个 token 的位置。
- 最后 `view(B, T, D)` 拍平回去。直观：每个 token 现在看到的是「来自所有 token 的同一通道块」，用接近 0 计算量完成跨 token 信息混合。
- 约束 `d_model % T == 0` 就是这步切分的需要。

### 7.3 FFN + 残差
- `LayerNorm → fc1(D→4D) → GELU → dropout → fc2(4D→D)`：标准 Transformer FFN。
- `Q_boost = Q + Q_e`：残差 **基于原始 Q**（不是 mixed 之后的），保留输入信息。
- `post_norm`：堆叠多层时稳定输出量级。


In [8]:
# === baseline train/model.py 第 315-412 行 ===
class RankMixerBlock(nn.Module):
    """HyFormer Query Boosting block.

    Performs three steps:
    1. Token Mixing: Parameter-free tensor reshaping.
    2. Per-token FFN: Shared-parameter feedforward network.
    3. Residual connection: Q_boost = Q + Q_e.

    Constraint: d_model must be divisible by n_total in 'full' mode.
    """

    def __init__(
        self,
        d_model: int,
        n_total: int,  # T = Nq + Nns
        hidden_mult: int = 4,
        dropout: float = 0.0,
        mode: str = 'full'  # 'full' | 'ffn_only' | 'none'
    ) -> None:
        super().__init__()
        self.T = n_total
        self.D = d_model
        self.mode = mode

        if mode == 'none':
            # Pure identity mapping, no submodules created
            return

        if mode == 'full':
            if d_model % n_total != 0:
                raise ValueError(
                    f"d_model={d_model} must be divisible by T={n_total} for token mixing."
                )
            self.d_sub = d_model // n_total

        # Per-token FFN (shared parameters) — used by both 'full' and 'ffn_only'
        self.norm = nn.LayerNorm(d_model)
        self.fc1 = nn.Linear(d_model, d_model * hidden_mult)
        self.fc2 = nn.Linear(d_model * hidden_mult, d_model)
        self.dropout = nn.Dropout(dropout)
        # Post-LN after residual to stabilize stacked block outputs
        self.post_norm = nn.LayerNorm(d_model)

    def token_mixing(self, Q: torch.Tensor) -> torch.Tensor:
        """Performs parameter-free token mixing via reshape and transpose.

        Steps:
        1. Splits channels into T subspaces: (B, T, D) -> (B, T, T, d_sub).
        2. Swaps token and subspace axes: (B, token, h, d_sub) -> (B, h, token, d_sub).
        3. Flattens back: (B, T, D).

        Args:
            Q: (B, T, D)

        Returns:
            Mixed tensor of shape (B, T, D).
        """
        B, T, D = Q.shape

        # (B, T, D) -> (B, T, T, d_sub)
        Q_split = Q.view(B, T, self.T, self.d_sub)

        # (B, token, h, d_sub) -> (B, h, token, d_sub)
        Q_rewired = Q_split.transpose(1, 2).contiguous()

        # (B, T, T, d_sub) -> (B, T, D)
        Q_hat = Q_rewired.view(B, T, D)
        return Q_hat

    def forward(self, Q: torch.Tensor) -> torch.Tensor:
        """Applies query boosting: token mixing, FFN, and residual connection.

        Args:
            Q: (B, T, D) where T = Nq + Nns.

        Returns:
            Boosted tensor of shape (B, T, D).
        """
        if self.mode == 'none':
            return Q

        # Token Mixing (parameter-free rewire) or identity
        if self.mode == 'full':
            Q_hat = self.token_mixing(Q)
        else:  # 'ffn_only'
            Q_hat = Q

        # Per-token FFN
        x = self.norm(Q_hat)
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        Q_e = self.fc2(x)

        # Residual from original Q
        Q_boost = Q + Q_e
        Q_boost = self.post_norm(Q_boost)
        return Q_boost


## Section 8 · `MultiSeqQueryGenerator` —— 为每个序列独立生成 Q tokens

PCVRHyFormer 有 S 个序列域（如 click、ad_click、conv 等），每个域要有自己的 Q tokens 去 cross-attn 该域的 KV。

### 8.1 设计思路
对每个序列 `i`：
- `GlobalInfo_i = Concat(NS_flat, MeanPool(Seq_i))`：把所有 NS token 拍平 + 该序列做 mean pooling，得到 `(B, (M+1)*D)` 的全局摘要。
- `Q_i = [FFN_{i,1}(GlobalInfo_i), …, FFN_{i,N}(GlobalInfo_i)]`：N 个独立 FFN 把全局摘要映射成 N 个 Q tokens。

### 8.2 实现细节
- `query_ffns_per_seq[i][q]`：嵌套 ModuleList，外层 S 个序列、内层 Nq 个 query；每个 FFN = `Linear → SiLU → Linear → LayerNorm`。
- `valid_mask = ~seq_padding_masks[i]`：True = 真实 token；mean pool 时只对真实位置求和再除以真实长度（用 `clamp(min=1)` 防止 0 序列除 0）。
- `global_info_norm`：对 `(M+1)*D` 这个高维 concat 加 LayerNorm，避免量级爆炸。
- 返回 list[S]，每个元素 `(B, Nq, D)` 的 q_tokens；后续会在 HyFormer block 内分别走 cross-attn。


In [9]:
# === baseline train/model.py 第 415-494 行 ===
class MultiSeqQueryGenerator(nn.Module):
    """Multi-sequence query generation module.

    Generates Q tokens independently for each sequence:
    For each sequence i:
        GlobalInfo_i = Concat(F1..FM, MeanPool(Seq_i))
        Q_i = [FFN_{i,1}(GlobalInfo_i), ..., FFN_{i,N}(GlobalInfo_i)]
    """

    def __init__(
        self,
        d_model: int,
        num_ns: int,
        num_queries: int,
        num_sequences: int,
        hidden_mult: int = 4
    ) -> None:
        super().__init__()
        self.num_queries = num_queries
        self.num_sequences = num_sequences
        self.d_model = d_model

        global_info_dim = (num_ns + 1) * d_model

        # LayerNorm on global_info to prevent gradient explosion from large-dim concat
        self.global_info_norm = nn.LayerNorm(global_info_dim)

        # Each sequence has N independent FFNs
        self.query_ffns_per_seq = nn.ModuleList([
            nn.ModuleList([
                nn.Sequential(
                    nn.Linear(global_info_dim, d_model * hidden_mult),
                    nn.SiLU(),
                    nn.Linear(d_model * hidden_mult, d_model),
                    nn.LayerNorm(d_model),
                )
                for _ in range(num_queries)
            ])
            for _ in range(num_sequences)
        ])

    def forward(
        self,
        ns_tokens: torch.Tensor,
        seq_tokens_list: list,
        seq_padding_masks: list
    ) -> list:
        """Generates query tokens for each sequence.

        Args:
            ns_tokens: (B, M, D), shared NS tokens.
            seq_tokens_list: List of (B, L_i, D) tensors, length S.
            seq_padding_masks: List of (B, L_i) masks, length S. True
                indicates padding.

        Returns:
            List of (B, Nq, D) query token tensors, length S.
        """
        B = ns_tokens.shape[0]
        ns_flat = ns_tokens.view(B, -1)  # (B, M*D)

        q_tokens_list = []
        for i in range(self.num_sequences):
            # MeanPool(Seq_i)
            valid_mask = ~seq_padding_masks[i]  # True = valid
            valid_mask_expanded = valid_mask.unsqueeze(-1).float()  # (B, L_i, 1)
            seq_sum = (seq_tokens_list[i] * valid_mask_expanded).sum(dim=1)  # (B, D)
            seq_count = valid_mask_expanded.sum(dim=1).clamp(min=1)  # (B, 1)
            seq_pooled = seq_sum / seq_count  # (B, D)

            # GlobalInfo_i = Concat(NS_flat, seq_pooled_i)
            global_info = torch.cat([ns_flat, seq_pooled], dim=-1)  # (B, (M+1)*D)
            global_info = self.global_info_norm(global_info)

            # Generate N query tokens
            queries = [ffn(global_info) for ffn in self.query_ffns_per_seq[i]]
            q_tokens = torch.stack(queries, dim=1)  # (B, Nq, D)
            q_tokens_list.append(q_tokens)

        return q_tokens_list


## Section 9 · `SwiGLUEncoder` —— 无注意力序列编码器

最轻量的序列编码方案：完全不做 token 间交互，每个位置独立过 `SwiGLU`。
- 适合：序列已经被前层处理过，不需要再做 token mixing；想省算力跑大 batch。
- 接口：`(x, key_padding_mask, **kwargs)`，**kwargs 吸收 `rope_cos/rope_sin`（当 baseline 在 block 里把它们传过来时不会报错）。
- 返回 `(out, key_padding_mask)`：把原 mask 一起回传，因为后面 cross-attn 还要用。


In [10]:
# === baseline train/model.py 第 502-541 行 ===
class SwiGLUEncoder(nn.Module):
    """Efficient attention-free sequence encoder.

    Structure: x + Dropout(SwiGLU(LN(x))).
    """

    def __init__(
        self,
        d_model: int,
        hidden_mult: int = 4,
        dropout: float = 0.0
    ) -> None:
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.swiglu = SwiGLU(d_model, hidden_mult)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: Optional[torch.Tensor] = None,
        **kwargs
    ) -> torch.Tensor:
        """Applies the SwiGLU encoder with residual connection.

        Args:
            x: (B, L, D)
            key_padding_mask: (B, L), True indicates padding. Not used by
                this encoder variant.
            **kwargs: Absorbs rope_cos/rope_sin and other unused parameters.

        Returns:
            Tuple of (output tensor of shape (B, L, D), key_padding_mask).
        """
        residual = x
        x = self.norm(x)
        x = self.swiglu(x)
        x = self.dropout(x)
        x = residual + x
        return x, key_padding_mask


## Section 10 · `TransformerEncoder` —— 标准 Transformer 层

经典 Pre-LN Transformer 编码层：`x + Attn(LN(x)) + FFN(LN(·))`，配合 RoPE。
- `self_attn`：rope_on_q=True 的 self-attention（Q=K=V=x）。
- `ffn`：`Linear→GELU→Dropout→Linear→Dropout` 的 4× FFN。
- 两个残差通路都 pre-LN，最后输出 mask 不变（self-attn 不改长度）。
- `seq_encoder_type='transformer'` 时每个序列域用一个独立的本类实例。


In [11]:
# === baseline train/model.py 第 544-614 行 ===
class TransformerEncoder(nn.Module):
    """High-capacity sequence encoder with self-attention and RoPE.

    Structure: Standard Transformer Encoder Layer (Pre-LN).
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        hidden_mult: int = 4,
        dropout: float = 0.0
    ) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.self_attn = RoPEMultiheadAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout,
            rope_on_q=True,
        )

        hidden_dim = d_model * hidden_mult
        self.ffn = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model),
            nn.Dropout(dropout)
        )

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: Optional[torch.Tensor] = None,
        rope_cos: Optional[torch.Tensor] = None,
        rope_sin: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Applies one Transformer encoder layer.

        Args:
            x: (B, L, D)
            key_padding_mask: (B, L), True indicates padding positions.
            rope_cos: (1, L, head_dim), RoPE cosine values.
            rope_sin: (1, L, head_dim), RoPE sine values.

        Returns:
            Tuple of (output tensor of shape (B, L, D), key_padding_mask).
        """
        # Self-Attention (Pre-LN) with RoPE
        residual = x
        x = self.norm1(x)
        x, _ = self.self_attn(
            query=x,
            key=x,
            value=x,
            key_padding_mask=key_padding_mask,
            rope_cos=rope_cos,
            rope_sin=rope_sin,
        )
        x = residual + x

        # FFN (Pre-LN)
        residual = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = residual + x

        return x, key_padding_mask


## Section 11 · `LongerEncoder` —— top-K 压缩编码器

为「超长序列」设计的自适应编码器：当 `L > top_k` 时做 cross-attn 压缩；当 `L ≤ top_k` 时退化成 self-attn。**多 block 串联时第一块自动压到 top_k，后续块在 top_k 上做 self-attn**。

### 11.1 _gather_top_k —— 提取最近 top_k 个有效 token
- `valid_len = (~mask).sum(1)`：每条样本的真实长度。
- `actual_k = min(valid_len, top_k)`、`start_pos = valid_len - actual_k`：取每条样本「最近 top_k 个有效位置」。
- `indices = start_pos[:, None] + arange(top_k)`：为每个 batch 算出 gather 索引；用 `clamp(0, L-1)` 防越界。
- 用 `torch.gather` 取出 token，并构造新 mask：前 `(top_k - actual_k)` 位是 padding。
- `position_indices`：返回原始位置索引（cross-attn 时给 Q 侧 RoPE 用）。

### 11.2 forward —— 长度自适应分流
**Cross-attn 模式（`L > top_k`）**：
1. `_gather_top_k` 取最近 top_k 个有效 token 当 Q。
2. `norm_q(q)`、`norm_kv(x)` 各自 pre-LN。
3. **Q 侧 RoPE 单独构造**：`q_pos_indices` 是不连续的位置（譬如样本里第 5、6、…、54 位），所以从 `rope_cos` 里 `gather` 出对应位置的 cos/sin → `(B, top_k, head_dim)`。
4. SDPA 跑 cross-attn，`key_padding_mask` 还是原始 `(B, L)`，残差基于 `q`（top_k 子集）。

**Self-attn 模式（`L ≤ top_k`）**：
- 直接 self-attn；`causal=True` 时叠 `generate_square_subsequent_mask` 做下三角 mask。

最后 `FFN + residual + ffn_norm` 收尾，返回 `(out, new_mask)`，把 mask 一并向后传，便于下游 block 接力。


In [12]:
# === baseline train/model.py 第 616-808 行 ===
class LongerEncoder(nn.Module):
    """Top-K compressed sequence encoder.

    Adapts behavior based on input length:
    - L > top_k (first MultiSeqHyFormerBlock): Cross Attention.
      Q = latest top_k tokens, K/V = all seq tokens -> output (B, top_k, D).
    - L <= top_k (subsequent MultiSeqHyFormerBlocks): Self Attention.
      Q = K = V = top_k tokens -> output (B, top_k, D).

    Causal mask is only applied among top_k tokens (self-attention layers);
    the first cross-attention layer does not use a causal mask since Q and K
    have different lengths.

    Returns (output, new_key_padding_mask) so downstream can update the mask.
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        top_k: int = 50,
        hidden_mult: int = 4,
        dropout: float = 0.0,
        causal: bool = False
    ) -> None:
        super().__init__()
        self.top_k = top_k
        self.causal = causal

        # Pre-LN for attention
        self.norm_q = nn.LayerNorm(d_model)
        self.norm_kv = nn.LayerNorm(d_model)

        # Shared RoPEMHA for both cross and self attention
        self.attn = RoPEMultiheadAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout,
            rope_on_q=True,
        )

        # FFN (Pre-LN + residual)
        self.ffn_norm = nn.LayerNorm(d_model)
        hidden_dim = d_model * hidden_mult
        self.ffn = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model),
            nn.Dropout(dropout)
        )

    def _gather_top_k(
        self,
        x: torch.Tensor,
        key_padding_mask: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Selects the latest top_k valid tokens from each sample.

        Args:
            x: (B, L, D)
            key_padding_mask: (B, L), True indicates padding.

        Returns:
            top_k_tokens: (B, top_k, D)
            new_padding_mask: (B, top_k), True indicates padding.
            position_indices: (B, top_k), original position index for each
                selected token, used for Q-side RoPE.
        """
        B, L, D = x.shape
        device = x.device

        # Valid lengths per sample
        valid_len = (~key_padding_mask).sum(dim=1)  # (B,)

        # Start position for each sample: max(valid_len - top_k, 0)
        actual_k = torch.clamp(valid_len, max=self.top_k)  # (B,)
        start_pos = valid_len - actual_k  # (B,)

        # Build gather indices: (B, top_k)
        offsets = torch.arange(self.top_k, device=device).unsqueeze(0).expand(B, -1)  # (B, top_k)
        indices = start_pos.unsqueeze(1) + offsets  # (B, top_k)

        # For samples with valid_len < top_k, early indices may exceed valid range;
        # clamp to [0, L-1] and handle via mask below
        indices = torch.clamp(indices, min=0, max=L - 1)

        # Gather: (B, top_k, D)
        indices_expanded = indices.unsqueeze(-1).expand(-1, -1, D)  # (B, top_k, D)
        top_k_tokens = torch.gather(x, dim=1, index=indices_expanded)

        # New padding mask: first (top_k - actual_k) positions are padding
        new_valid_len = actual_k  # (B,)
        pad_count = self.top_k - new_valid_len  # (B,)
        pos_indices = torch.arange(self.top_k, device=device).unsqueeze(0)  # (1, top_k)
        new_padding_mask = pos_indices < pad_count.unsqueeze(1)  # (B, top_k)

        # Zero out tokens at padding positions
        top_k_tokens = top_k_tokens * (~new_padding_mask).unsqueeze(-1).float()

        # position_indices for Q-side RoPE
        position_indices = indices  # (B, top_k)

        return top_k_tokens, new_padding_mask, position_indices

    def forward(
        self,
        x: torch.Tensor,
        key_padding_mask: Optional[torch.Tensor] = None,
        rope_cos: Optional[torch.Tensor] = None,
        rope_sin: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Applies the LongerEncoder with adaptive cross/self attention.

        Args:
            x: (B, L, D), sequence tokens.
            key_padding_mask: (B, L), True indicates padding.
            rope_cos: (1, L, head_dim), RoPE cosine values (length must cover
                original sequence length L).
            rope_sin: (1, L, head_dim), RoPE sine values.

        Returns:
            output: (B, top_k, D), compressed sequence.
            new_key_padding_mask: (B, top_k), updated padding mask.
        """
        B, L, D = x.shape

        if L > self.top_k:
            # === Cross Attention mode (first MultiSeqHyFormerBlock) ===
            # 1. Extract latest top_k tokens as query
            q, new_mask, q_pos_indices = self._gather_top_k(x, key_padding_mask)

            # 2. Pre-LN
            q_normed = self.norm_q(q)
            kv_normed = self.norm_kv(x)

            # 3. Build Q-side RoPE cos/sin by gathering from global cos/sin at top_k positions
            q_rope_cos = None
            q_rope_sin = None
            if rope_cos is not None and rope_sin is not None:
                # rope_cos: (1, L_max, head_dim), q_pos_indices: (B, top_k)
                head_dim = rope_cos.shape[2]
                # Expand to batch dimension
                cos_expanded = rope_cos.expand(B, -1, -1)  # (B, L_max, head_dim)
                sin_expanded = rope_sin.expand(B, -1, -1)
                idx = q_pos_indices.unsqueeze(-1).expand(-1, -1, head_dim)  # (B, top_k, head_dim)
                q_rope_cos = torch.gather(cos_expanded, 1, idx)  # (B, top_k, head_dim)
                q_rope_sin = torch.gather(sin_expanded, 1, idx)

            # 4. Cross Attention (no causal mask since Q and K have different lengths)
            attn_out, _ = self.attn(
                query=q_normed,
                key=kv_normed,
                value=kv_normed,
                key_padding_mask=key_padding_mask,  # Original (B, L) mask
                rope_cos=rope_cos,
                rope_sin=rope_sin,
                q_rope_cos=q_rope_cos,
                q_rope_sin=q_rope_sin,
            )
            out = q + attn_out  # Residual based on q
        else:
            # === Self Attention mode (subsequent MultiSeqHyFormerBlocks) ===
            new_mask = key_padding_mask

            # Pre-LN (Q and KV share norm_q)
            x_normed = self.norm_q(x)

            # Causal mask
            attn_mask = None
            if self.causal:
                attn_mask = nn.Transformer.generate_square_subsequent_mask(
                    L, device=x.device
                )

            attn_out, _ = self.attn(
                query=x_normed,
                key=x_normed,
                value=x_normed,
                key_padding_mask=key_padding_mask,
                attn_mask=attn_mask,
                rope_cos=rope_cos,
                rope_sin=rope_sin,
            )
            out = x + attn_out

        # FFN (Pre-LN + residual)
        residual = out
        out = self.ffn_norm(out)
        out = self.ffn(out)
        out = residual + out

        return out, new_mask


## Section 12 · `create_sequence_encoder` —— 编码器工厂

通过字符串选编码器，避免在 `MultiSeqHyFormerBlock` 里写 if/elif。
- `'swiglu'`：极轻量、无 attention（最快但表达力弱）。
- `'transformer'`：标准 Transformer，**baseline 默认推荐**。
- `'longer'`：长序列首选（top-K 压缩）。


In [13]:
# === baseline train/model.py 第 811-842 行 ===
def create_sequence_encoder(
    encoder_type: str,
    d_model: int,
    num_heads: int = 4,
    hidden_mult: int = 4,
    dropout: float = 0.0,
    top_k: int = 50,
    causal: bool = False
) -> nn.Module:
    """Creates a sequence encoder of the specified type.

    Args:
        encoder_type: One of 'swiglu', 'transformer', or 'longer'.
        d_model: Model dimension.
        num_heads: Number of attention heads (used by transformer/longer).
        hidden_mult: FFN expansion multiplier.
        dropout: Dropout rate.
        top_k: Compression length for LongerEncoder (only used by longer).
        causal: Whether to use causal mask in LongerEncoder (only used by
            longer).

    Returns:
        A sequence encoder module.
    """
    if encoder_type == 'swiglu':
        return SwiGLUEncoder(d_model, hidden_mult, dropout)
    elif encoder_type == 'transformer':
        return TransformerEncoder(d_model, num_heads, hidden_mult, dropout)
    elif encoder_type == 'longer':
        return LongerEncoder(d_model, num_heads, top_k, hidden_mult, dropout, causal)
    else:
        raise ValueError(f"Unknown encoder type: {encoder_type}")


## Section 13 · `MultiSeqHyFormerBlock` —— 多序列 HyFormer 主块

PCVRHyFormer 的核心计算单元，每个 block 内部分 5 步处理 S 个序列：

### 13.1 init 三大子模块
- `seq_encoders`：S 个独立的序列编码器（如 TransformerEncoder × S）。
- `cross_attns`：S 个独立的 CrossAttention，让每个序列的 Q 去 cross-attn 该序列的 KV。
- `mixer`：1 个共享 RankMixerBlock，n_total = `Nq*S + Nns`（所有序列 Q + 所有 NS）。

### 13.2 forward 5 步
1. **Sequence Evolution**：`for i in S: next_seqs[i] = seq_encoders[i](seq_tokens[i], mask[i], rope)`。注意 `LongerEncoder` 会顺便把 mask 改了（top_k 压缩后 mask 也变短）。
2. **Query Decoding**：`for i in S: decoded_qs[i] = cross_attns[i](q_tokens[i], next_seqs[i], next_masks[i], rope)`。每个序列的 Q 只看自己的 KV。
3. **Token Fusion**：`combined = cat(decoded_qs + [ns_tokens], dim=1)` → `(B, Nq*S + Nns, D)`。
4. **Query Boosting**：`boosted = mixer(combined)`；RankMixer 的 token mixing 在这里跨「序列 Q」与「NS token」混合信息。
5. **Split back**：把 `boosted` 按 `Nq*S | Nns` 切回去，更新 `q_tokens_list` 和 `ns_tokens`。

返回 `(next_q_list, next_ns, next_seqs, next_masks)`：4 个状态全部更新，喂给下一个 block。


In [14]:
# === baseline train/model.py 第 850-980 行 ===
class MultiSeqHyFormerBlock(nn.Module):
    """Multi-sequence HyFormer block.

    Each of the S sequences independently performs Sequence Evolution and
    Query Decoding, then all Q tokens and shared NS tokens are merged for
    joint Query Boosting.
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        num_queries: int,
        num_ns: int,
        num_sequences: int,
        seq_encoder_type: str = 'swiglu',
        hidden_mult: int = 4,
        dropout: float = 0.0,
        top_k: int = 50,
        causal: bool = False,
        rank_mixer_mode: str = 'full'
    ) -> None:
        super().__init__()
        self.num_sequences = num_sequences
        self.num_queries = num_queries
        self.num_ns = num_ns

        # Independent sequence encoder per sequence
        self.seq_encoders = nn.ModuleList([
            create_sequence_encoder(
                encoder_type=seq_encoder_type,
                d_model=d_model,
                num_heads=num_heads,
                hidden_mult=hidden_mult,
                dropout=dropout,
                top_k=top_k,
                causal=causal
            )
            for _ in range(num_sequences)
        ])

        # Independent cross-attention per sequence
        self.cross_attns = nn.ModuleList([
            CrossAttention(
                d_model=d_model,
                num_heads=num_heads,
                dropout=dropout,
                ln_mode='pre'
            )
            for _ in range(num_sequences)
        ])

        # RankMixer: input token count = Nq * S + Nns
        n_total = num_queries * num_sequences + num_ns
        self.mixer = RankMixerBlock(
            d_model=d_model,
            n_total=n_total,
            hidden_mult=hidden_mult,
            dropout=dropout,
            mode=rank_mixer_mode
        )

    def forward(
        self,
        q_tokens_list: list,
        ns_tokens: torch.Tensor,
        seq_tokens_list: list,
        seq_padding_masks: list,
        rope_cos_list: Optional[List[torch.Tensor]] = None,
        rope_sin_list: Optional[List[torch.Tensor]] = None,
    ) -> Tuple[list, torch.Tensor, list, list]:
        """Processes one multi-sequence HyFormer block step.

        Args:
            q_tokens_list: List of (B, Nq, D) tensors, length S.
            ns_tokens: (B, Nns, D)
            seq_tokens_list: List of (B, L_i, D) tensors, length S.
            seq_padding_masks: List of (B, L_i) masks, length S.
            rope_cos_list: List of (1, L_i, head_dim) tensors, length S.
            rope_sin_list: List of (1, L_i, head_dim) tensors, length S.

        Returns:
            A tuple (next_q_list, next_ns, next_seq_list, next_masks), where
            next_q_list is a list of (B, Nq, D) updated query tensors,
            next_ns is (B, Nns, D) updated non-sequence tokens,
            next_seq_list is a list of (B, L_i', D) encoded sequence tensors,
            and next_masks is a list of (B, L_i') updated padding masks.
        """
        S = self.num_sequences
        Nq = self.num_queries

        # 1. Independent Sequence Evolution per sequence
        next_seqs = []
        next_masks = []
        for i in range(S):
            rc = rope_cos_list[i] if rope_cos_list is not None else None
            rs = rope_sin_list[i] if rope_sin_list is not None else None
            result = self.seq_encoders[i](
                seq_tokens_list[i], seq_padding_masks[i],
                rope_cos=rc, rope_sin=rs,
            )
            next_seq_i, mask_i = result
            next_seqs.append(next_seq_i)
            next_masks.append(mask_i)

        # 2. Independent Query Decoding per sequence
        decoded_qs = []
        for i in range(S):
            rc = rope_cos_list[i] if rope_cos_list is not None else None
            rs = rope_sin_list[i] if rope_sin_list is not None else None
            decoded_q_i = self.cross_attns[i](
                q_tokens_list[i], next_seqs[i], next_masks[i],
                rope_cos=rc, rope_sin=rs,
            )
            decoded_qs.append(decoded_q_i)

        # 3. Token Fusion: concatenate all decoded_q + ns_tokens
        combined = torch.cat(decoded_qs + [ns_tokens], dim=1)  # (B, Nq*S + Nns, D)

        # 4. Query Boosting
        boosted = self.mixer(combined)  # (B, Nq*S + Nns, D)

        # 5. Split back into per-sequence Q and NS
        next_q_list = []
        offset = 0
        for i in range(S):
            next_q_list.append(boosted[:, offset:offset + Nq, :])
            offset += Nq
        next_ns = boosted[:, offset:, :]

        return next_q_list, next_ns, next_seqs, next_masks


## Section 14 · `GroupNSTokenizer` —— 按 group 一格一 token

NS（Non-Sequence）token 的第一种构造方式：**每个 group 输出一个 token**。

### 14.1 init 关键
- `feature_specs: List[(vocab_size, offset, length)]`：每个 fid 在 concat 后的 `int_feats` 里的位置和槽位长度。`length=1` 是单值，`length>1` 是多值（如多类别 tag）。
- `groups: List[List[int]]`：把 fid 按业务语义分组（user 侧分 N 个 group → N 个 token）。
- 跳过条件：`vs <= 0`（无 vocab 信息）或 `emb_skip_threshold > 0 且 vs > 阈值`（过高基数特征跳过 Embedding，节省显存）。
- 跳过的 fid 用 `None` 占位，`_emb_index[i]=-1` 表示没有 Embedding 表，forward 时输出零向量。

### 14.2 forward
对每个 group：
- 对每个 fid：
  - 单值 → `embedding(int_feats[:, offset]).long()`，结果 `(B, emb_dim)`。
  - 多值 → 切片 `(B, length)` 查表 → `(B, length, emb_dim)` → 用 mask 过滤 padding=0 → mean pool → `(B, emb_dim)`。
- 把 group 内所有 fid 的 embedding 沿最后一维 concat → `(B, num_fids*emb_dim)`。
- 过 `Linear → LayerNorm → SiLU` 投影到 `d_model`，`unsqueeze(1)` 变 `(B, 1, D)`。
- 最后 cat 所有 group → `(B, num_groups, D)`。


In [15]:
# === baseline train/model.py 第 988-1067 行 ===
class GroupNSTokenizer(nn.Module):
    """NS tokenizer used by ns_tokenizer_type='group'.

    Groups discrete features by fid, applies shared embedding with mean
    pooling per multi-valued feature, then projects each group to a single
    NS token (one token per group).
    """

    def __init__(self, feature_specs: List[Tuple[int, int, int]],
                 groups: List[List[int]], emb_dim: int, d_model: int,
                 emb_skip_threshold: int = 0) -> None:
        super().__init__()
        self.feature_specs = feature_specs
        self.groups = groups
        self.emb_dim = emb_dim
        self.emb_skip_threshold = emb_skip_threshold

        # One embedding table per fid (None if skipped by emb_skip_threshold
        # or if vocab_size <= 0 / no vocab info).
        embs = []
        for vs, offset, length in feature_specs:
            skip = int(vs) <= 0 or (emb_skip_threshold > 0 and int(vs) > emb_skip_threshold)
            if skip:
                embs.append(None)
            else:
                embs.append(nn.Embedding(int(vs) + 1, emb_dim, padding_idx=0))
        self.embs = nn.ModuleList([e for e in embs if e is not None])
        # Map from fid index to position in self.embs (or -1 if filtered)
        self._emb_index = []
        real_idx = 0
        for e in embs:
            if e is not None:
                self._emb_index.append(real_idx)
                real_idx += 1
            else:
                self._emb_index.append(-1)

        # Per-group projection: num_fids_in_group * emb_dim -> d_model (with LayerNorm)
        self.group_projs = nn.ModuleList([
            nn.Sequential(
                nn.Linear(len(group) * emb_dim, d_model),
                nn.LayerNorm(d_model),
            )
            for group in groups
        ])

    def forward(self, int_feats: torch.Tensor) -> torch.Tensor:
        """Embeds and projects grouped discrete features into NS tokens.

        Args:
            int_feats: (B, total_int_dim), concatenated integer features.

        Returns:
            Tokens of shape (B, num_groups, D).
        """
        tokens = []
        for group, proj in zip(self.groups, self.group_projs):
            fid_embs = []
            for fid_idx in group:
                vs, offset, length = self.feature_specs[fid_idx]
                emb_real_idx = self._emb_index[fid_idx]
                if emb_real_idx == -1:
                    # Filtered high-cardinality feature: output zero vector
                    fid_emb = int_feats.new_zeros(int_feats.shape[0], self.emb_dim)
                else:
                    emb_layer = self.embs[emb_real_idx]
                    if length == 1:
                        # Single-value feature: direct lookup
                        fid_emb = emb_layer(int_feats[:, offset].long())  # (B, emb_dim)
                    else:
                        # Multi-value feature: lookup then mean pooling (ignoring padding=0)
                        vals = int_feats[:, offset:offset + length].long()  # (B, length)
                        emb_all = emb_layer(vals)  # (B, length, emb_dim)
                        mask = (vals != 0).float().unsqueeze(-1)  # (B, length, 1)
                        count = mask.sum(dim=1).clamp(min=1)  # (B, 1)
                        fid_emb = (emb_all * mask).sum(dim=1) / count  # (B, emb_dim)
                fid_embs.append(fid_emb)
            cat_emb = torch.cat(fid_embs, dim=-1)  # (B, num_fids*emb_dim)
            tokens.append(F.silu(proj(cat_emb)).unsqueeze(1))  # (B, 1, D)
        return torch.cat(tokens, dim=1)  # (B, num_groups, D)


## Section 15 · `RankMixerNSTokenizer` —— cat → split → project（baseline 默认）

NS token 的第二种构造方式（对应 RankMixer 论文）：把所有 group 内所有 fid 的 embedding **全部 cat 成一根长向量，等分成 num_ns_tokens 段，每段独立 project 到 d_model**。这样 `num_ns_tokens` 可以**自由选择**，不再受 group 数限制。

### 15.1 init
- 跟 GroupNSTokenizer 一样按 fid 建 Embedding（共享 `emb_skip_threshold` 跳过逻辑）。
- 算总维度 `total_emb_dim = sum_groups(len(group)) * emb_dim`。
- `chunk_dim = ceil(total_emb_dim / num_ns_tokens)`，再补 padding 让 `chunk_dim * num_ns_tokens >= total_emb_dim`。这样切分时刚好等分。
- 为每个 chunk 建独立的 `Linear(chunk_dim → d_model) + LayerNorm`。

### 15.2 forward
1. 按 group 顺序把所有 fid 的 embedding `(B, emb_dim)` append 进 `all_embs` list。
2. concat 成 `(B, total_emb_dim)`，必要时 `F.pad` 到 `padded_total_dim`。
3. `cat_emb.split(chunk_dim, dim=-1)` 切成 `num_ns_tokens` 段，每段过对应 `token_proj` + `SiLU`，`unsqueeze(1)` 后 cat → `(B, num_ns_tokens, d_model)`。

### 为什么 baseline 默认选 rankmixer？
group 模式下 `num_ns` 由 group 数决定，灵活度低；rankmixer 模式下 `num_ns` 是超参，可以调整 T = `num_queries*num_sequences + num_ns` 来满足 `d_model % T == 0` 的约束。


In [16]:
# === baseline train/model.py 第 1070-1189 行 ===
class RankMixerNSTokenizer(nn.Module):
    """NS Tokenizer following the RankMixer paper's approach.

    All group embedding vectors are concatenated into a single long vector,
    then equally split into num_ns_tokens segments, each projected to d_model.
    This allows num_ns_tokens to be chosen freely (independent of group count).
    """

    def __init__(
        self,
        feature_specs: List[Tuple[int, int, int]],
        groups: List[List[int]],
        emb_dim: int,
        d_model: int,
        num_ns_tokens: int,
        emb_skip_threshold: int = 0,
    ) -> None:
        """Initializes RankMixerNSTokenizer.

        Args:
            feature_specs: [(vocab_size, offset, length), ...] per feature.
            groups: List of feature index groups (defines semantic ordering).
            emb_dim: Embedding dimension per feature.
            d_model: Output token dimension.
            num_ns_tokens: Number of NS tokens to produce (T segments).
            emb_skip_threshold: Skip embedding for features with vocab > threshold.
        """
        super().__init__()
        self.feature_specs = feature_specs
        self.groups = groups
        self.emb_dim = emb_dim
        self.num_ns_tokens = num_ns_tokens
        self.emb_skip_threshold = emb_skip_threshold

        # One embedding table per fid (None if skipped by emb_skip_threshold
        # or if vocab_size <= 0 / no vocab info).
        embs = []
        for vs, offset, length in feature_specs:
            skip = int(vs) <= 0 or (emb_skip_threshold > 0 and int(vs) > emb_skip_threshold)
            if skip:
                embs.append(None)
            else:
                embs.append(nn.Embedding(int(vs) + 1, emb_dim, padding_idx=0))
        self.embs = nn.ModuleList([e for e in embs if e is not None])
        # Map from fid index to position in self.embs (or -1 if filtered)
        self._emb_index = []
        real_idx = 0
        for e in embs:
            if e is not None:
                self._emb_index.append(real_idx)
                real_idx += 1
            else:
                self._emb_index.append(-1)

        # Compute total embedding dim: sum of all fids across all groups
        total_num_fids = sum(len(g) for g in groups)
        total_emb_dim = total_num_fids * emb_dim

        # Pad total_emb_dim to be divisible by num_ns_tokens
        self.chunk_dim = math.ceil(total_emb_dim / num_ns_tokens)
        self.padded_total_dim = self.chunk_dim * num_ns_tokens
        self._pad_size = self.padded_total_dim - total_emb_dim

        # Per-chunk projection: chunk_dim -> d_model with LayerNorm
        self.token_projs = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.chunk_dim, d_model),
                nn.LayerNorm(d_model),
            )
            for _ in range(num_ns_tokens)
        ])

        logging.info(
            f"RankMixerNSTokenizer: {total_num_fids} fids, "
            f"total_emb_dim={total_emb_dim}, chunk_dim={self.chunk_dim}, "
            f"num_ns_tokens={num_ns_tokens}, pad={self._pad_size}"
        )

    def forward(self, int_feats: torch.Tensor) -> torch.Tensor:
        """Embeds all features, concatenates, splits, and projects.

        Args:
            int_feats: (B, total_int_dim) concatenated integer features.

        Returns:
            (B, num_ns_tokens, d_model) tensor.
        """
        # 1. Embed all fids in group order → flat cat
        all_embs = []
        for group in self.groups:
            for fid_idx in group:
                vs, offset, length = self.feature_specs[fid_idx]
                emb_real_idx = self._emb_index[fid_idx]
                if emb_real_idx == -1:
                    fid_emb = int_feats.new_zeros(int_feats.shape[0], self.emb_dim)
                else:
                    emb_layer = self.embs[emb_real_idx]
                    if length == 1:
                        fid_emb = emb_layer(int_feats[:, offset].long())
                    else:
                        vals = int_feats[:, offset:offset + length].long()
                        emb_all = emb_layer(vals)
                        mask = (vals != 0).float().unsqueeze(-1)
                        count = mask.sum(dim=1).clamp(min=1)
                        fid_emb = (emb_all * mask).sum(dim=1) / count
                all_embs.append(fid_emb)

        cat_emb = torch.cat(all_embs, dim=-1)  # (B, total_emb_dim)

        # 2. Pad if needed
        if self._pad_size > 0:
            cat_emb = F.pad(cat_emb, (0, self._pad_size))  # (B, padded_total_dim)

        # 3. Split into num_ns_tokens chunks and project each
        chunks = cat_emb.split(self.chunk_dim, dim=-1)  # list of (B, chunk_dim)
        tokens = []
        for chunk, proj in zip(chunks, self.token_projs):
            tokens.append(F.silu(proj(chunk)).unsqueeze(1))  # (B, 1, d_model)

        return torch.cat(tokens, dim=1)  # (B, num_ns_tokens, d_model)


## Section 16 · `PCVRHyFormer` —— 主模型

整个 baseline 的总装类。逻辑上分 3 段，分别用 markdown 讲解；
**代码受 Jupyter cell 边界限制必须放在一格**，否则方法定义因缩进会跨 cell 报错。

### 16.1 `__init__`（baseline 第 1199-1452 行）
- `seq_domains = sorted(seq_vocab_sizes.keys())`：用 sorted 保证 domain 顺序确定，避免 ckpt 加载混乱。
- `ns_tokenizer_type='rankmixer'`（默认）走 `RankMixerNSTokenizer`，否则 `'group'` 走 `GroupNSTokenizer`。
- dense feature 可选：`user_dense_dim>0` 时单独建 Linear+LayerNorm 投影成 1 个 NS token。
- **关键约束**：`T = num_queries*num_sequences + num_ns`，要求 `rank_mixer_mode='full'` 时 `d_model % T == 0`。报错信息会列出所有合法 T 值。
- 时间桶 Embedding：`num_time_buckets=65`（含 padding=0），加在 token embedding 上。
- `RotaryEmbedding(dim=head_dim)`：注意是 head_dim，不是 d_model；多头各自旋转。
- `output_proj`：所有序列的 Q tokens cat 后拍平 `(B, Nq*S*D)` → `Linear → d_model`。
- `clsfier`：`Linear(d, d) → LN → SiLU → Dropout → Linear(d, action_num)`，PCVR 时 `action_num=1`。

### 16.2 私有方法（baseline 第 1454-1632 行）
- `_init_params`：所有 Embedding 用 Xavier normal 初始化，并把 `padding_idx=0` 那一行强制清零。
- `reinit_high_cardinality_params(threshold)`：MEDA 重置——只重置 `vocab_size > threshold` 的高基数表。返回被重置参数的 `data_ptr` 集合，trainer 用它来重置对应优化器状态。
- `get_sparse_params` / `get_dense_params`：通过 `data_ptr` 区分 Embedding 参数（喂 Adagrad）与其他（喂 AdamW）。
- `_embed_seq_domain`：序列域 Embedding 流水线——按 fid 查表 + concat + Linear+LN+GELU + 加时间桶 embedding。**id 类特征**（`vs > seq_id_threshold`）训练时额外过 `seq_id_emb_dropout`（dropout 加倍）。
- `_make_padding_mask`：用 `arange(L) >= seq_len` 一次性广播出 mask。
- `_run_multi_seq_blocks`：对 N 个 HyFormer block 顺序走，每步先算 RoPE cos/sin，传给 block；最后把所有序列的 Q 拍平 → `output_proj` → `(B, D)`。

### 16.3 `forward` / `predict`（baseline 第 1634-1714 行）
5 步：
1. **NS tokens**：user_int → `user_ns_tokenizer` → `(B, num_user_ns, D)`；item_int 同理；user_dense / item_dense（如有）各自投影成 1 个 token；最后 cat 成 `(B, num_ns, D)`。
2. **序列 Embedding**：对每个 domain 调 `_embed_seq_domain` 得到 `(B, L, D)`，并算 padding mask。
3. **生成 Q tokens**：`query_generator(ns_tokens, seq_tokens_list, masks)` → `list[S]`，每个 `(B, Nq, D)`。
4. **HyFormer Block 堆叠 + 输出投影**：`_run_multi_seq_blocks` → `(B, D)`。
5. **分类器**：`clsfier(output)` → `(B, action_num)`，logits 给 trainer 做 BCE/Focal loss。

`predict` = `forward` + `apply_dropout=False` + 额外返回 `output_embedding`，可作向量召回。


In [17]:
# === baseline train/model.py 第 1192-1714 行（PCVRHyFormer 完整类） ===
class PCVRHyFormer(nn.Module):
    """PCVRHyFormer model for post-click conversion rate prediction.

    Combines MultiSeqHyFormerBlock and MultiSeqQueryGenerator to process
    multiple input sequences with non-sequence features.
    """

    def __init__(
        self,
        # Data schema
        user_int_feature_specs: List[Tuple[int, int, int]],
        item_int_feature_specs: List[Tuple[int, int, int]],
        user_dense_dim: int,
        item_dense_dim: int,
        seq_vocab_sizes: "dict[str, List[int]]",  # {domain: [vocab_size_per_fid, ...]}
        # NS grouping config (grouped by fid index)
        user_ns_groups: List[List[int]],
        item_ns_groups: List[List[int]],
        # Model hyperparameters
        d_model: int = 64,
        emb_dim: int = 64,
        num_queries: int = 1,
        num_hyformer_blocks: int = 2,
        num_heads: int = 4,
        seq_encoder_type: str = 'transformer',
        hidden_mult: int = 4,
        dropout_rate: float = 0.01,
        seq_top_k: int = 50,
        seq_causal: bool = False,
        action_num: int = 1,
        num_time_buckets: int = 65,
        rank_mixer_mode: str = 'full',
        use_rope: bool = False,
        rope_base: float = 10000.0,
        emb_skip_threshold: int = 0,
        seq_id_threshold: int = 10000,
        # NS tokenizer variant
        ns_tokenizer_type: str = 'rankmixer',
        user_ns_tokens: int = 0,
        item_ns_tokens: int = 0,
    ) -> None:
        super().__init__()

        self.d_model = d_model
        self.emb_dim = emb_dim
        self.action_num = action_num
        self.num_queries = num_queries
        self.seq_domains = sorted(seq_vocab_sizes.keys())  # deterministic order
        self.num_sequences = len(self.seq_domains)
        self.num_time_buckets = num_time_buckets
        self.rank_mixer_mode = rank_mixer_mode
        self.use_rope = use_rope
        self.emb_skip_threshold = emb_skip_threshold
        self.seq_id_threshold = seq_id_threshold
        self.ns_tokenizer_type = ns_tokenizer_type

        # ================== NS Tokens Construction ==================

        if ns_tokenizer_type == 'group':
            # Original: one NS token per group
            self.user_ns_tokenizer = GroupNSTokenizer(
                feature_specs=user_int_feature_specs,
                groups=user_ns_groups,
                emb_dim=emb_dim,
                d_model=d_model,
                emb_skip_threshold=emb_skip_threshold,
            )
            num_user_ns = len(user_ns_groups)

            self.item_ns_tokenizer = GroupNSTokenizer(
                feature_specs=item_int_feature_specs,
                groups=item_ns_groups,
                emb_dim=emb_dim,
                d_model=d_model,
                emb_skip_threshold=emb_skip_threshold,
            )
            num_item_ns = len(item_ns_groups)
        elif ns_tokenizer_type == 'rankmixer':
            # RankMixer paper style: all embeddings cat → split → project
            # 0 means auto: fall back to group count
            if user_ns_tokens <= 0:
                user_ns_tokens = len(user_ns_groups)
            if item_ns_tokens <= 0:
                item_ns_tokens = len(item_ns_groups)
            self.user_ns_tokenizer = RankMixerNSTokenizer(
                feature_specs=user_int_feature_specs,
                groups=user_ns_groups,
                emb_dim=emb_dim,
                d_model=d_model,
                num_ns_tokens=user_ns_tokens,
                emb_skip_threshold=emb_skip_threshold,
            )
            num_user_ns = user_ns_tokens

            self.item_ns_tokenizer = RankMixerNSTokenizer(
                feature_specs=item_int_feature_specs,
                groups=item_ns_groups,
                emb_dim=emb_dim,
                d_model=d_model,
                num_ns_tokens=item_ns_tokens,
                emb_skip_threshold=emb_skip_threshold,
            )
            num_item_ns = item_ns_tokens
        else:
            raise ValueError(f"Unknown ns_tokenizer_type: {ns_tokenizer_type}")

        # User dense feature projection (if available)
        self.has_user_dense = user_dense_dim > 0
        if self.has_user_dense:
            self.user_dense_proj = nn.Sequential(
                nn.Linear(user_dense_dim, d_model),
                nn.LayerNorm(d_model),
            )

        # Item dense feature projection (if available)
        self.has_item_dense = item_dense_dim > 0
        if self.has_item_dense:
            self.item_dense_proj = nn.Sequential(
                nn.Linear(item_dense_dim, d_model),
                nn.LayerNorm(d_model),
            )

        # Total NS token count
        self.num_ns = (num_user_ns + (1 if self.has_user_dense else 0)
                       + num_item_ns + (1 if self.has_item_dense else 0))

        # ================== Check d_model % T == 0 constraint (full mode only) ==================
        T = num_queries * self.num_sequences + self.num_ns
        if rank_mixer_mode == 'full' and d_model % T != 0:
            valid_T_values = [t for t in range(1, d_model + 1) if d_model % t == 0]
            raise ValueError(
                f"d_model={d_model} must be divisible by T=num_queries*num_sequences+num_ns="
                f"{num_queries}*{self.num_sequences}+{self.num_ns}={T}. "
                f"Valid T values for d_model={d_model}: {valid_T_values}"
            )

        # ================== Seq Tokens Embedding ==================
        # seq_id_threshold decides which features inside the seq tokenizer are
        # treated as id features (they receive extra dropout). It is fully
        # independent of emb_skip_threshold (which skips Embedding creation).
        self.seq_id_emb_dropout = nn.Dropout(dropout_rate * 2)

        def _make_seq_embs(vocab_sizes):
            """Create embedding list, returning None for features skipped via
            emb_skip_threshold or with no vocab info (vs<=0)."""
            embs_raw = []
            for vs in vocab_sizes:
                skip = int(vs) <= 0 or (emb_skip_threshold > 0 and int(vs) > emb_skip_threshold)
                if skip:
                    embs_raw.append(None)
                else:
                    embs_raw.append(nn.Embedding(int(vs) + 1, emb_dim, padding_idx=0))
            module_list = nn.ModuleList([e for e in embs_raw if e is not None])
            # Map from position index to real index in module_list (-1 if skipped)
            index_map = []
            real_idx = 0
            for e in embs_raw:
                if e is not None:
                    index_map.append(real_idx)
                    real_idx += 1
                else:
                    index_map.append(-1)
            is_id = [int(vs) > seq_id_threshold for vs in vocab_sizes]
            return module_list, index_map, is_id

        # ================== Dynamic Sequence Embeddings ==================
        self._seq_embs = nn.ModuleDict()
        self._seq_emb_index = {}    # domain -> index_map
        self._seq_is_id = {}        # domain -> is_id list
        self._seq_vocab_sizes = {}  # domain -> vocab_sizes list
        self._seq_proj = nn.ModuleDict()

        for domain in self.seq_domains:
            vs = seq_vocab_sizes[domain]
            embs, idx_map, is_id = _make_seq_embs(vs)
            self._seq_embs[domain] = embs
            self._seq_emb_index[domain] = idx_map
            self._seq_is_id[domain] = is_id
            self._seq_vocab_sizes[domain] = vs
            self._seq_proj[domain] = nn.Sequential(
                nn.Linear(len(vs) * emb_dim, d_model),
                nn.LayerNorm(d_model),
            )

        # ================== Time Interval Bucket Embedding (optional) ==================
        if num_time_buckets > 0:
            self.time_embedding = nn.Embedding(num_time_buckets, d_model, padding_idx=0)

        # ================== HyFormer Components ==================
        # MultiSeqQueryGenerator
        self.query_generator = MultiSeqQueryGenerator(
            d_model=d_model,
            num_ns=self.num_ns,
            num_queries=num_queries,
            num_sequences=self.num_sequences,
            hidden_mult=hidden_mult,
        )

        # MultiSeqHyFormerBlock stack
        self.blocks = nn.ModuleList([
            MultiSeqHyFormerBlock(
                d_model=d_model,
                num_heads=num_heads,
                num_queries=num_queries,
                num_ns=self.num_ns,
                num_sequences=self.num_sequences,
                seq_encoder_type=seq_encoder_type,
                hidden_mult=hidden_mult,
                dropout=dropout_rate,
                top_k=seq_top_k,
                causal=seq_causal,
                rank_mixer_mode=rank_mixer_mode,
            )
            for _ in range(num_hyformer_blocks)
        ])

        # ================== RoPE ==================
        if use_rope:
            head_dim = d_model // num_heads
            self.rotary_emb = RotaryEmbedding(dim=head_dim, base=rope_base)
        else:
            self.rotary_emb = None

        # Output projection
        self.output_proj = nn.Sequential(
            nn.Linear(num_queries * self.num_sequences * d_model, d_model),
            nn.LayerNorm(d_model),
        )

        # Dropout
        self.emb_dropout = nn.Dropout(dropout_rate)

        # Classifier
        self.clsfier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
            nn.SiLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(d_model, action_num)
        )

        # Initialize parameters
        self._init_params()

        # Log emb_skip_threshold filtering stats
        if emb_skip_threshold > 0:
            def _count_filtered(vocab_sizes, emb_index):
                filtered = sum(1 for idx in emb_index if idx == -1)
                return filtered, len(vocab_sizes)
            for domain in self.seq_domains:
                f, t = _count_filtered(self._seq_vocab_sizes[domain], self._seq_emb_index[domain])
                if f > 0:
                    logging.info(f"emb_skip_threshold={emb_skip_threshold}: {domain} skipped {f}/{t} features")
            for name, tokenizer in [
                ("user_ns", self.user_ns_tokenizer),
                ("item_ns", self.item_ns_tokenizer),
            ]:
                f = sum(1 for idx in tokenizer._emb_index if idx == -1)
                t = len(tokenizer._emb_index)
                if f > 0:
                    logging.info(f"emb_skip_threshold={emb_skip_threshold}: {name} skipped {f}/{t} features")

    def _init_params(self) -> None:
        """Applies Xavier initialization to all embedding weights."""
        for domain in self.seq_domains:
            for emb in self._seq_embs[domain]:
                nn.init.xavier_normal_(emb.weight.data)
                emb.weight.data[0, :] = 0

        for tokenizer in [self.user_ns_tokenizer, self.item_ns_tokenizer]:
            for emb in tokenizer.embs:
                nn.init.xavier_normal_(emb.weight.data)
                emb.weight.data[0, :] = 0

        if self.num_time_buckets > 0:
            nn.init.xavier_normal_(self.time_embedding.weight.data)
            self.time_embedding.weight.data[0, :] = 0

    def reinit_high_cardinality_params(
        self, cardinality_threshold: int = 10000
    ) -> "set[int]":
        """Reinitializes only high-cardinality embeddings.

        Preserves low-cardinality and time feature embeddings.

        Args:
            cardinality_threshold: Only embeddings with vocab_size exceeding
                this value are reinitialized.

        Returns:
            A set of data_ptr() values for reinitialized parameters.
        """
        reinit_count = 0
        skip_count = 0
        reinit_ptrs = set()

        for emb_list, vocab_sizes, emb_index in [
            (self._seq_embs[d], self._seq_vocab_sizes[d], self._seq_emb_index[d])
            for d in self.seq_domains
        ]:
            for i, vs in enumerate(vocab_sizes):
                real_idx = emb_index[i]
                if real_idx == -1:
                    # Skipped by emb_skip_threshold, no embedding to reinit
                    continue
                emb = emb_list[real_idx]
                if int(vs) > cardinality_threshold:
                    nn.init.xavier_normal_(emb.weight.data)
                    emb.weight.data[0, :] = 0
                    reinit_ptrs.add(emb.weight.data_ptr())
                    reinit_count += 1
                else:
                    skip_count += 1

        for tokenizer, specs in [
            (self.user_ns_tokenizer, self.user_ns_tokenizer.feature_specs),
            (self.item_ns_tokenizer, self.item_ns_tokenizer.feature_specs),
        ]:
            for i, (vs, offset, length) in enumerate(specs):
                real_idx = tokenizer._emb_index[i]
                if real_idx == -1:
                    continue
                emb = tokenizer.embs[real_idx]
                if int(vs) > cardinality_threshold:
                    nn.init.xavier_normal_(emb.weight.data)
                    emb.weight.data[0, :] = 0
                    reinit_ptrs.add(emb.weight.data_ptr())
                    reinit_count += 1
                else:
                    skip_count += 1

        # time_embedding is always preserved
        if self.num_time_buckets > 0:
            skip_count += 1

        logging.info(f"Re-initialized {reinit_count} high-cardinality Embeddings "
                     f"(vocab>{cardinality_threshold}), kept {skip_count}")
        return reinit_ptrs

    def get_sparse_params(self) -> List[nn.Parameter]:
        """Returns all embedding table parameters (optimized with Adagrad)."""
        sparse_params = set()
        for module in self.modules():
            if isinstance(module, nn.Embedding):
                sparse_params.add(module.weight.data_ptr())
        return [p for p in self.parameters() if p.data_ptr() in sparse_params]

    def get_dense_params(self) -> List[nn.Parameter]:
        """Returns all non-embedding parameters (optimized with AdamW)."""
        sparse_ptrs = {p.data_ptr() for p in self.get_sparse_params()}
        return [p for p in self.parameters() if p.data_ptr() not in sparse_ptrs]

    def _embed_seq_domain(
        self,
        seq: torch.Tensor,
        sideinfo_embs: nn.ModuleList,
        proj: nn.Module,
        is_id: List[bool],
        emb_index: List[int],
        time_bucket_ids: torch.Tensor,
    ) -> torch.Tensor:
        """Embeds a sequence domain by concatenating sideinfo embeddings and projecting to d_model."""
        B, S, L = seq.shape
        emb_list = []
        for i in range(S):
            real_idx = emb_index[i] if i < len(emb_index) else -1
            if real_idx == -1:
                # Feature skipped by emb_skip_threshold: output zero vector
                emb_list.append(seq.new_zeros(B, L, self.emb_dim, dtype=torch.float))
            else:
                emb = sideinfo_embs[real_idx]
                e = emb(seq[:, i, :])  # (B, L, emb_dim)
                if is_id[i] and self.training:
                    e = self.seq_id_emb_dropout(e)
                emb_list.append(e)
        cat_emb = torch.cat(emb_list, dim=-1)  # (B, L, S*emb_dim)
        token_emb = F.gelu(proj(cat_emb))  # (B, L, D)

        # Add time bucket embedding (all-zero ids produce zero vectors via padding_idx=0)
        if self.num_time_buckets > 0:
            token_emb = token_emb + self.time_embedding(time_bucket_ids)

        return token_emb

    def _make_padding_mask(
        self, seq_len: torch.Tensor, max_len: int
    ) -> torch.Tensor:
        """Generates a padding mask from sequence lengths."""
        device = seq_len.device
        idx = torch.arange(max_len, device=device).unsqueeze(0)  # (1, max_len)
        return idx >= seq_len.unsqueeze(1)  # (B, max_len)

    def _run_multi_seq_blocks(
        self,
        q_tokens_list: list,
        ns_tokens: torch.Tensor,
        seq_tokens_list: list,
        seq_masks_list: list,
        apply_dropout: bool = True
    ) -> torch.Tensor:
        """Runs the multi-sequence block stack with dropout and output projection."""
        if apply_dropout:
            q_tokens_list = [self.emb_dropout(q) for q in q_tokens_list]
            ns_tokens = self.emb_dropout(ns_tokens)
            seq_tokens_list = [self.emb_dropout(s) for s in seq_tokens_list]

        curr_qs = q_tokens_list
        curr_ns = ns_tokens
        curr_seqs = seq_tokens_list
        curr_masks = seq_masks_list

        for block in self.blocks:
            # Precompute RoPE cos/sin for each sequence
            rope_cos_list = None
            rope_sin_list = None
            if self.rotary_emb is not None:
                rope_cos_list = []
                rope_sin_list = []
                device = curr_seqs[0].device
                for seq_i in curr_seqs:
                    seq_len = seq_i.shape[1]
                    cos, sin = self.rotary_emb(seq_len, device)
                    rope_cos_list.append(cos)
                    rope_sin_list.append(sin)

            curr_qs, curr_ns, curr_seqs, curr_masks = block(
                q_tokens_list=curr_qs,
                ns_tokens=curr_ns,
                seq_tokens_list=curr_seqs,
                seq_padding_masks=curr_masks,
                rope_cos_list=rope_cos_list,
                rope_sin_list=rope_sin_list,
            )

        # Output: concatenate all sequences' Q tokens then project via MLP
        B = curr_qs[0].shape[0]
        all_q = torch.cat(curr_qs, dim=1)  # (B, Nq*S, D)
        output = all_q.view(B, -1)  # (B, Nq*S*D)
        output = self.output_proj(output)  # (B, D)

        return output

    def forward(self, inputs: ModelInput) -> torch.Tensor:
        """Runs the forward pass of the PCVRHyFormer model."""
        # 1. NS tokens: grouped projection
        user_ns = self.user_ns_tokenizer(inputs.user_int_feats)   # (B, num_user_groups, D)
        item_ns = self.item_ns_tokenizer(inputs.item_int_feats)   # (B, num_item_groups, D)

        ns_parts = [user_ns]
        if self.has_user_dense:
            user_dense_tok = F.silu(self.user_dense_proj(inputs.user_dense_feats)).unsqueeze(1)  # (B, 1, D)
            ns_parts.append(user_dense_tok)
        ns_parts.append(item_ns)
        if self.has_item_dense:
            item_dense_tok = F.silu(self.item_dense_proj(inputs.item_dense_feats)).unsqueeze(1)  # (B, 1, D)
            ns_parts.append(item_dense_tok)

        ns_tokens = torch.cat(ns_parts, dim=1)  # (B, num_ns, D)

        # 2. Embed each sequence domain (dynamic)
        seq_tokens_list = []
        seq_masks_list = []
        for domain in self.seq_domains:
            tokens = self._embed_seq_domain(
                inputs.seq_data[domain],
                self._seq_embs[domain], self._seq_proj[domain],
                self._seq_is_id[domain], self._seq_emb_index[domain],
                inputs.seq_time_buckets[domain])
            seq_tokens_list.append(tokens)
            mask = self._make_padding_mask(inputs.seq_lens[domain], inputs.seq_data[domain].shape[2])
            seq_masks_list.append(mask)

        # 3. Generate independent Q tokens per sequence via MultiSeqQueryGenerator
        q_tokens_list = self.query_generator(ns_tokens, seq_tokens_list, seq_masks_list)

        # 4. Dropout + MultiSeqHyFormerBlock stack + output projection
        output = self._run_multi_seq_blocks(
            q_tokens_list, ns_tokens, seq_tokens_list, seq_masks_list,
            apply_dropout=self.training
        )

        # 5. Classifier
        logits = self.clsfier(output)  # (B, action_num)
        return logits

    def predict(self, inputs: ModelInput) -> Tuple[torch.Tensor, torch.Tensor]:
        """Runs inference without dropout, returning both logits and embeddings."""
        # Reuses forward logic but without dropout
        user_ns = self.user_ns_tokenizer(inputs.user_int_feats)
        item_ns = self.item_ns_tokenizer(inputs.item_int_feats)

        ns_parts = [user_ns]
        if self.has_user_dense:
            user_dense_tok = F.silu(self.user_dense_proj(inputs.user_dense_feats)).unsqueeze(1)
            ns_parts.append(user_dense_tok)
        ns_parts.append(item_ns)
        if self.has_item_dense:
            item_dense_tok = F.silu(self.item_dense_proj(inputs.item_dense_feats)).unsqueeze(1)
            ns_parts.append(item_dense_tok)

        ns_tokens = torch.cat(ns_parts, dim=1)

        seq_tokens_list = []
        seq_masks_list = []
        for domain in self.seq_domains:
            tokens = self._embed_seq_domain(
                inputs.seq_data[domain],
                self._seq_embs[domain], self._seq_proj[domain],
                self._seq_is_id[domain], self._seq_emb_index[domain],
                inputs.seq_time_buckets[domain])
            seq_tokens_list.append(tokens)
            mask = self._make_padding_mask(inputs.seq_lens[domain], inputs.seq_data[domain].shape[2])
            seq_masks_list.append(mask)

        q_tokens_list = self.query_generator(ns_tokens, seq_tokens_list, seq_masks_list)

        output = self._run_multi_seq_blocks(
            q_tokens_list, ns_tokens, seq_tokens_list, seq_masks_list,
            apply_dropout=False
        )

        logits = self.clsfier(output)
        return logits, output


## Section 17 · 数据准备（双分支）

> **设计决策**：baseline 原版 [`train/train.py`](train/train.py) 假设你已经有真实 `*.parquet` + 真实 `schema.json`，
> 它**不包含任何数据生成 / schema 推断逻辑**。本 notebook 为了在「LFS 没拉到」的情况下也能端到端跑通，加了一段兜底脚手架。

### 两条路径

```
                    ┌── 真实数据存在? ──┐
                    │                   │
       是（真 parquet）                否（LFS pointer / 缺失）
                    │                   │
                    ▼                   ▼
           走 baseline 原版分支    走 notebook 兜底分支
        ────────────────────       ────────────────────
        17.A 检测 LFS 指针         17.A 检测 LFS 指针
                                   17.B 列名清单（合成数据需要）
                                   17.C 生成合成 parquet
        17.D 推断 schema.json      17.D 推断 schema.json（同样的代码）
```

| 子节 | 标注 | 说明 |
|---|---|---|
| 17.A | ✅ 通用 | LFS pointer 检测 + 决定走哪条分支 |
| 17.B | 🔧 仅兜底分支 | 合成数据用的列名清单（按 [`sample_data/README_data.md`](sample_data/README_data.md) 抄） |
| 17.C | 🔧 仅兜底分支 | 用 numpy.random 生成假 parquet（**与 baseline 无关**） |
| 17.D | ✅ 通用 | 从 parquet 头部推断 schema.json（baseline 没给官方文件，必须自己造） |

### 怎么强制走原版分支？

把 17.A 里的 `FORCE_REAL_DATA = False` 改成 `True`：检测到 LFS pointer 时直接抛错，不再 fallback。
适合「我已经 `git lfs pull` 拉到真实数据，不希望被静默 fallback 蒙骗」的场景。

### 真实数据怎么来？
1. 本地：`cd taac-codes && git lfs install && git lfs pull`
2. Kaggle：拉完后重新打包 zip 上传成 dataset 新版本（详见之前对话）
3. 完整训练集：从 TAAC 竞赛官网下载 `demo_10w.parquet` / 全量数据


In [ ]:
# === 17.A · ✅ 通用：LFS 指针检测 + 分支决策 ===
import pyarrow as pa
import pyarrow.parquet as pq

REAL_PARQUET = SAMPLE_DIR / 'demo_1000.parquet'

# ⚙️ 切换开关：True = 严格模式（缺真实数据直接报错）；False = 自动 fallback 合成
FORCE_REAL_DATA = False

def is_lfs_pointer(p: Path) -> bool:
    """判断 parquet 是不是 Git LFS 指针文件（133 字节的纯文本，不是真实二进制）"""
    if not p.exists() or p.stat().st_size > 4096:
        return False
    try:
        head = p.read_bytes()[:200].decode('utf-8', errors='ignore')
        return head.startswith('version https://git-lfs')
    except Exception:
        return False

use_real_data = REAL_PARQUET.exists() and not is_lfs_pointer(REAL_PARQUET)

print(f'real parquet exists : {REAL_PARQUET.exists()}')
print(f'is LFS pointer      : {is_lfs_pointer(REAL_PARQUET)}')
if REAL_PARQUET.exists():
    sz_kb = REAL_PARQUET.stat().st_size / 1024
    print(f'parquet size        : {sz_kb:.1f} KB  ({"真实数据" if sz_kb > 4 else "LFS pointer"})')

if FORCE_REAL_DATA and not use_real_data:
    raise RuntimeError(
        f'FORCE_REAL_DATA=True 但 {REAL_PARQUET} 不是真实数据。'
        f'请先 `git lfs pull` 或关闭 FORCE_REAL_DATA。'
    )

print(f'\n==> 决策：{"🟢 走 baseline 原版分支（真实数据）" if use_real_data else "🟡 走 notebook 兜底分支（合成数据）"}')
print(f'    （后续 17.B/17.C 会{"跳过" if use_real_data else "执行"}，17.D 都会跑）')


In [ ]:
# === 17.B · 🔧 仅兜底分支：列名清单（按 sample_data/README_data.md 严格对齐） ===
# baseline 原版没有这部分；这是合成数据生成时用到的 fid 列表，从 README_data.md 抄出来的
if not use_real_data:
    USER_INT_SCALAR_FIDS = [1, 3, 4] + list(range(48, 60)) + [82, 86] + list(range(92, 110))   # 35 列
    USER_INT_ARRAY_FIDS  = [15, 60] + list(range(62, 67)) + [80] + list(range(89, 92))         # 11 列
    USER_DENSE_FIDS      = list(range(61, 67)) + [87] + list(range(89, 92))                    # 10 列
    ITEM_INT_SCALAR_FIDS = list(range(5, 11)) + [12, 13, 16, 81] + list(range(83, 86))         # 13 列
    ITEM_INT_ARRAY_FIDS  = [11]                                                                # 1 列
    DOMAIN_A_FIDS = list(range(38, 47))                  # 9 列
    DOMAIN_B_FIDS = list(range(67, 80)) + [88]           # 14 列
    DOMAIN_C_FIDS = list(range(27, 38)) + [47]           # 12 列
    DOMAIN_D_FIDS = list(range(17, 27))                  # 10 列
    DOMAINS = {
        'seq_a': ('domain_a_seq', DOMAIN_A_FIDS),
        'seq_b': ('domain_b_seq', DOMAIN_B_FIDS),
        'seq_c': ('domain_c_seq', DOMAIN_C_FIDS),
        'seq_d': ('domain_d_seq', DOMAIN_D_FIDS),
    }
    total_cols = (5 + len(USER_INT_SCALAR_FIDS) + len(USER_INT_ARRAY_FIDS)
                    + len(USER_DENSE_FIDS) + len(ITEM_INT_SCALAR_FIDS) + len(ITEM_INT_ARRAY_FIDS)
                    + sum(len(f) for _, f in DOMAINS.values()))
    print(f'🔧 兜底：列名清单已就绪，总共 {total_cols} 列（应为 120）')
else:
    print('🟢 跳过：使用真实数据，无需合成数据列名清单')


In [ ]:
# === 17.C · 🔧 仅兜底分支：生成合成 parquet（baseline 无此步） ===
# 用 numpy.random 编造一份格式合法、但「特征 X 与 label Y 完全独立」的假数据。
# 因此模型在合成数据上 valid AUC 必然 ≈ 0.5（这是数据问题，不是模型问题）。
# 想要真 AUC：拉 LFS 真实数据后重跑。
rng = np.random.default_rng(42)
N_ROWS = 1000
VOCAB_SCALAR = 200
VOCAB_ARRAY  = 500
VOCAB_SEQ    = 1000
DENSE_DIM    = 8

def _scalar_int_col(vocab):
    return rng.integers(1, vocab, size=N_ROWS, dtype=np.int64)
def _array_int_col(vocab, lo=1, hi=5):
    return [rng.integers(1, vocab, size=int(rng.integers(lo, hi + 1)), dtype=np.int64).tolist()
            for _ in range(N_ROWS)]
def _array_float_col(dim):
    return [rng.standard_normal(dim).astype(np.float32).tolist() for _ in range(N_ROWS)]
def _seq_int_col(vocab, lo=5, hi=30):
    return [rng.integers(1, vocab, size=int(rng.integers(lo, hi + 1)), dtype=np.int64).tolist()
            for _ in range(N_ROWS)]
def _seq_ts_col(base_ts, lo=5, hi=30):
    out = []
    for b in base_ts:
        L = int(rng.integers(lo, hi + 1))
        deltas = rng.integers(60, 86400, size=L, dtype=np.int64)
        out.append((b - np.cumsum(deltas)).astype(np.int64).tolist())
    return out

if not use_real_data:
    cols = {}
    cols['user_id']    = rng.integers(1, 50_000, size=N_ROWS, dtype=np.int64)
    cols['item_id']    = rng.integers(1, 50_000, size=N_ROWS, dtype=np.int64)
    cols['label_type'] = rng.choice([0, 1, 2, 3], size=N_ROWS, p=[0.4, 0.2, 0.3, 0.1]).astype(np.int32)
    base_ts = rng.integers(1_700_000_000, 1_710_000_000, size=N_ROWS, dtype=np.int64)
    cols['label_time'] = base_ts.copy()
    cols['timestamp']  = base_ts

    for fid in USER_INT_SCALAR_FIDS:  cols[f'user_int_feats_{fid}']  = _scalar_int_col(VOCAB_SCALAR)
    for fid in USER_INT_ARRAY_FIDS:   cols[f'user_int_feats_{fid}']  = _array_int_col(VOCAB_ARRAY)
    for fid in USER_DENSE_FIDS:       cols[f'user_dense_feats_{fid}']= _array_float_col(DENSE_DIM)
    for fid in ITEM_INT_SCALAR_FIDS:  cols[f'item_int_feats_{fid}']  = _scalar_int_col(VOCAB_SCALAR)
    for fid in ITEM_INT_ARRAY_FIDS:   cols[f'item_int_feats_{fid}']  = _array_int_col(VOCAB_ARRAY)

    domain_meta = {}
    for dname, (prefix, fids) in DOMAINS.items():
        ts_fid = fids[-1]
        for fid in fids:
            col_name = f'{prefix}_{fid}'
            cols[col_name] = (_seq_ts_col(base_ts) if fid == ts_fid else _seq_int_col(VOCAB_SEQ))
        domain_meta[dname] = {'prefix': prefix, 'ts_fid': ts_fid, 'fids': fids}

    arrays, names = [], []
    for k, v in cols.items():
        if k == 'label_type':
            arrays.append(pa.array(v, type=pa.int32()))
        elif isinstance(v, np.ndarray):
            arrays.append(pa.array(v, type=pa.int64()))
        elif isinstance(v[0], list) and len(v[0]) > 0 and isinstance(v[0][0], float):
            arrays.append(pa.array(v, type=pa.list_(pa.float32())))
        else:
            arrays.append(pa.array(v, type=pa.list_(pa.int64())))
        names.append(k)

    table = pa.Table.from_arrays(arrays, names=names)
    SYN_DATA_DIR = WORK_DIR / 'data'
    SYN_DATA_DIR.mkdir(exist_ok=True)
    syn_pq_path = SYN_DATA_DIR / 'demo_synth.parquet'
    pq.write_table(table, syn_pq_path, row_group_size=200)   # 5 row groups → 切 train/valid
    print(f'🔧 兜底：合成 parquet 已写入 {syn_pq_path}（{N_ROWS} 行 × {len(names)} 列，5 个 row group）')
    DATA_DIR = SYN_DATA_DIR
else:
    domain_meta = None
    DATA_DIR = SAMPLE_DIR
    print(f'🟢 原版：使用真实数据 {DATA_DIR}')


In [ ]:
# === 17.D · ✅ 通用：从 parquet 头部推断 schema.json ===
# baseline 原版要求你**自己**提供 schema.json（[`train/dataset.py`](train/dataset.py:672) 的 schema_path 参数），
# 但 sample_data/ 没附官方文件 → 必须自己写或推断。下面这段从真实/合成 parquet 头部 200 行扫一遍，自动推断：
# - 离散特征 vocab_size = 实际最大值 + 1
# - 数组特征 dim = 实际最大长度
# 推断结果存到 WORK_DIR/schema.json，传给 get_pcvr_data。
#
# ⚠️  如果将来官方发了 schema.json，应该跳过这段，直接：
#     schema_path = '/path/to/official/schema.json'
from collections import OrderedDict

pq_files = sorted(DATA_DIR.glob('*.parquet'))
assert pq_files, f'没找到 parquet 文件：{DATA_DIR}'

_pf = pq.ParquetFile(pq_files[0])
_col_types = {f.name: f.type for f in _pf.schema_arrow}

def _array_dim_and_vocab(col_name, max_rows=200):
    tbl = _pf.read(columns=[col_name]).slice(0, max_rows)
    arr = tbl.column(0)
    max_len, max_val = 1, 1
    for x in arr.to_pylist():
        if x is None: continue
        max_len = max(max_len, len(x))
        if len(x) > 0 and isinstance(x[0], (int, np.integer)):
            max_val = max(max_val, int(max(x)))
    return max_len, max_val + 1

def _scalar_vocab(col_name, max_rows=1000):
    tbl = _pf.read(columns=[col_name]).slice(0, max_rows)
    arr = tbl.column(0).to_numpy()
    return int(arr.max()) + 1 if len(arr) else 1

# 真实数据时从 parquet 列名自动发现 fid（不依赖 17.B 的列表）
def _discover_fids(prefix):
    fids = []
    for col in _col_types:
        if col.startswith(prefix + '_'):
            try: fids.append(int(col[len(prefix) + 1:]))
            except ValueError: pass
    return sorted(fids)

if use_real_data:
    user_int_scalar_fids_eff = []
    user_int_array_fids_eff  = []
    for fid in _discover_fids('user_int_feats'):
        name = f'user_int_feats_{fid}'
        # 区分 scalar/array：看 arrow type
        if pa.types.is_list(_col_types[name]):
            user_int_array_fids_eff.append(fid)
        else:
            user_int_scalar_fids_eff.append(fid)
    item_int_scalar_fids_eff = []
    item_int_array_fids_eff  = []
    for fid in _discover_fids('item_int_feats'):
        name = f'item_int_feats_{fid}'
        if pa.types.is_list(_col_types[name]):
            item_int_array_fids_eff.append(fid)
        else:
            item_int_scalar_fids_eff.append(fid)
    user_dense_fids_eff = _discover_fids('user_dense_feats')
    domains_eff = {}
    for dname, prefix in [('seq_a','domain_a_seq'),('seq_b','domain_b_seq'),
                          ('seq_c','domain_c_seq'),('seq_d','domain_d_seq')]:
        fids = _discover_fids(prefix)
        if fids:
            domains_eff[dname] = (prefix, fids)
else:
    user_int_scalar_fids_eff = USER_INT_SCALAR_FIDS
    user_int_array_fids_eff  = USER_INT_ARRAY_FIDS
    item_int_scalar_fids_eff = ITEM_INT_SCALAR_FIDS
    item_int_array_fids_eff  = ITEM_INT_ARRAY_FIDS
    user_dense_fids_eff      = USER_DENSE_FIDS
    domains_eff              = DOMAINS

schema = OrderedDict()
ui = []
for fid in user_int_scalar_fids_eff:
    name = f'user_int_feats_{fid}'
    if name in _col_types: ui.append([fid, _scalar_vocab(name), 1])
for fid in user_int_array_fids_eff:
    name = f'user_int_feats_{fid}'
    if name in _col_types:
        d, v = _array_dim_and_vocab(name); ui.append([fid, v, d])
schema['user_int'] = ui

ii = []
for fid in item_int_scalar_fids_eff:
    name = f'item_int_feats_{fid}'
    if name in _col_types: ii.append([fid, _scalar_vocab(name), 1])
for fid in item_int_array_fids_eff:
    name = f'item_int_feats_{fid}'
    if name in _col_types:
        d, v = _array_dim_and_vocab(name); ii.append([fid, v, d])
schema['item_int'] = ii

ud = []
for fid in user_dense_fids_eff:
    name = f'user_dense_feats_{fid}'
    if name in _col_types:
        d, _ = _array_dim_and_vocab(name); ud.append([fid, d])
schema['user_dense'] = ud

schema['seq'] = {}
for dname, (prefix, fids) in domains_eff.items():
    feat_list = []
    for fid in fids:
        name = f'{prefix}_{fid}'
        if name in _col_types:
            _, v = _array_dim_and_vocab(name); feat_list.append([fid, v])
    if feat_list:
        ts_fid = (domain_meta[dname]['ts_fid'] if (domain_meta and dname in domain_meta) else fids[-1])
        schema['seq'][dname] = {'prefix': prefix, 'ts_fid': ts_fid, 'features': feat_list}

schema_path = WORK_DIR / 'schema.json'
with open(schema_path, 'w', encoding='utf-8') as f:
    json.dump(schema, f, indent=2, ensure_ascii=False)

print(f'✓ schema.json 写入 {schema_path}')
print(f'  user_int  : {len(schema["user_int"])} 组')
print(f'  item_int  : {len(schema["item_int"])} 组')
print(f'  user_dense: {len(schema["user_dense"])} 组')
print(f'  seq 域    : {list(schema["seq"].keys())}')


## Section 18 · 实例化 PCVRHyFormer + 单 batch 烟囱测试

调 baseline 的 `get_pcvr_data` 拿 DataLoader（它内部读 schema.json 并打包 ModelInput）；
然后用前面 Section 16 中 **notebook 自定义** 的 `PCVRHyFormer` 类实例化，验证 forward 能跑通。

> 注意：trainer 内部 `from model import ModelInput` 会引用 baseline 的 ModelInput。我们自己也定义了同字段的 ModelInput，
> 但 NamedTuple 是 duck typing，trainer 构造的 baseline.ModelInput 实例传给我们的 forward 完全兼容。


In [22]:
# 18.A：构建 DataLoader
from utils import set_seed, create_logger
from dataset import get_pcvr_data, NUM_TIME_BUCKETS

set_seed(42)
create_logger(str(WORK_DIR / 'train.log'))

DATA_CFG = dict(
    data_dir=str(DATA_DIR),
    schema_path=str(schema_path),
    batch_size=64,
    valid_ratio=0.2,            # 5 row groups → 末尾 1 个做 valid（200 行）
    train_ratio=1.0,
    num_workers=0,              # macOS / notebook 推荐 0
    buffer_batches=4,
    seed=42,
    seq_max_lens={'seq_a': 32, 'seq_b': 32, 'seq_c': 64, 'seq_d': 64},
)

train_loader, valid_loader, pcvr_dataset = get_pcvr_data(**DATA_CFG)
print('NUM_TIME_BUCKETS =', NUM_TIME_BUCKETS)
print('train rows  :', pcvr_dataset.num_rows)
print('seq domains :', pcvr_dataset.seq_domains)


04/29/26 16:33:53 - 0:00:00 - Row Group split: 4 train (800 rows), 1 valid (200 rows)
04/29/26 16:33:53 - 0:00:00 - PCVRParquetDataset: 800 rows from 1 file(s), batch_size=64, buffer_batches=4, shuffle=True
04/29/26 16:33:53 - 0:00:00 - PCVRParquetDataset: 200 rows from 1 file(s), batch_size=64, buffer_batches=0, shuffle=False
04/29/26 16:33:53 - 0:00:00 - Parquet train: 800 rows, valid: 200 rows, batch_size=64, buffer_batches=4


NUM_TIME_BUCKETS = 64
train rows  : 800
seq domains : ['seq_a', 'seq_b', 'seq_c', 'seq_d']


In [23]:
# 18.B：把 FeatureSchema 转成模型需要的 feature_specs 格式（来自 baseline train/train.py）
def build_feature_specs(schema_obj, per_position_vocab_sizes):
    specs = []
    for fid, offset, length in schema_obj.entries:
        vs = max(per_position_vocab_sizes[offset:offset + length])
        specs.append((vs, offset, length))
    return specs

user_int_specs = build_feature_specs(pcvr_dataset.user_int_schema, pcvr_dataset.user_int_vocab_sizes)
item_int_specs = build_feature_specs(pcvr_dataset.item_int_schema, pcvr_dataset.item_int_vocab_sizes)

# fallback：每个特征独占 group（baseline 没有 ns_groups.json 时的默认行为）
user_ns_groups = [[i] for i in range(len(pcvr_dataset.user_int_schema.entries))]
item_ns_groups = [[i] for i in range(len(pcvr_dataset.item_int_schema.entries))]

MODEL_CFG = dict(
    user_int_feature_specs=user_int_specs,
    item_int_feature_specs=item_int_specs,
    user_dense_dim=pcvr_dataset.user_dense_schema.total_dim,
    item_dense_dim=pcvr_dataset.item_dense_schema.total_dim,
    seq_vocab_sizes=pcvr_dataset.seq_domain_vocab_sizes,
    user_ns_groups=user_ns_groups,
    item_ns_groups=item_ns_groups,
    d_model=64, emb_dim=64,
    num_queries=2, num_hyformer_blocks=2, num_heads=4,
    seq_encoder_type='transformer',
    hidden_mult=4, dropout_rate=0.1,
    seq_top_k=50, seq_causal=False,
    action_num=1,
    num_time_buckets=NUM_TIME_BUCKETS,
    rank_mixer_mode='full',
    use_rope=False, rope_base=10000.0,
    emb_skip_threshold=1_000_000,   # 极大 vocab 跳过 Embedding（节省显存）
    seq_id_threshold=10_000,        # 序列内大 vocab 列额外加 dropout
    ns_tokenizer_type='rankmixer',
    user_ns_tokens=5,
    item_ns_tokens=2,
)

# !!!! 关键：这里用的是【notebook 自定义】的 PCVRHyFormer 类（前面 Section 16 已构建），不是 from model import !!!!
model = PCVRHyFormer(**MODEL_CFG).to(DEVICE)

S = len(pcvr_dataset.seq_domains)
T = MODEL_CFG['num_queries'] * S + model.num_ns
print(f'PCVRHyFormer 已构建（来自 notebook 内的类定义）')
print(f'  num_sequences S = {S}')
print(f'  num_ns          = {model.num_ns}')
print(f'  T = num_queries*S + num_ns = {MODEL_CFG["num_queries"]}*{S} + {model.num_ns} = {T}')
print(f'  d_model % T = {MODEL_CFG["d_model"]} % {T} = {MODEL_CFG["d_model"] % T}  (must be 0)')
print(f'  total params = {sum(p.numel() for p in model.parameters()):,}')
print(f'  sparse (Emb) = {sum(p.numel() for p in model.get_sparse_params()):,}')
print(f'  dense        = {sum(p.numel() for p in model.get_dense_params()):,}')


04/29/26 16:33:53 - 0:00:00 - RankMixerNSTokenizer: 46 fids, total_emb_dim=2944, chunk_dim=589, num_ns_tokens=5, pad=1
04/29/26 16:33:53 - 0:00:00 - RankMixerNSTokenizer: 14 fids, total_emb_dim=896, chunk_dim=448, num_ns_tokens=2, pad=0


PCVRHyFormer 已构建（来自 notebook 内的类定义）
  num_sequences S = 4
  num_ns          = 8
  T = num_queries*S + num_ns = 2*4 + 8 = 16
  d_model % T = 64 % 16 = 0  (must be 0)
  total params = 6,074,689
  sparse (Emb) = 3,632,448
  dense        = 2,442,241


In [24]:
# 18.C：单 batch 烟囱测试
from model import ModelInput as BaselineModelInput   # baseline 版本（trainer 内部会用这个）

batch = next(iter(train_loader))
def to_dev(x):
    if torch.is_tensor(x): return x.to(DEVICE)
    if isinstance(x, dict): return {k: to_dev(v) for k, v in x.items()}
    return x

# pcvr_dataset 输出的 batch 字典里，序列字段是 domain 名，长度是 f"{domain}_len" 等
seq_data, seq_lens, seq_buckets = {}, {}, {}
for d in pcvr_dataset.seq_domains:
    seq_data[d]    = to_dev(batch[d])
    seq_lens[d]    = to_dev(batch[f'{d}_len'])
    seq_buckets[d] = to_dev(batch[f'{d}_time_bucket'])

inputs = BaselineModelInput(
    user_int_feats=to_dev(batch['user_int_feats']),
    item_int_feats=to_dev(batch['item_int_feats']),
    user_dense_feats=to_dev(batch['user_dense_feats']),
    item_dense_feats=to_dev(batch['item_dense_feats']),
    seq_data=seq_data, seq_lens=seq_lens, seq_time_buckets=seq_buckets,
)

model.eval()
with torch.no_grad():
    logits = model(inputs)
print(f'[smoke test] logits.shape = {tuple(logits.shape)}')
print(f'[smoke test] logits[:5]   = {logits[:5].cpu().numpy().ravel()}')
print('forward OK ✓')


[smoke test] logits.shape = (64, 1)
[smoke test] logits[:5]   = [-0.14016312 -0.09738667  0.08007687 -0.01937676 -0.07278189]
forward OK ✓


## Section 19 · 调用 baseline trainer 训练

直接复用 [`PCVRHyFormerRankingTrainer`](train/trainer.py)。它的接口跟模型完全解耦——只要模型有 `forward(ModelInput)`、`get_sparse_params()`、`get_dense_params()`、`reinit_high_cardinality_params(threshold)` 这 4 个方法就能接管。

训练器关键设计：
- **双优化器**：sparse params (Embedding) → Adagrad(lr=0.05)；dense params → AdamW(lr=1e-4)。
- **MEDA 重置**：每 epoch 末重置 `vocab_size > threshold` 的高基数 Embedding（演示中关闭）。
- **EarlyStopping**：监控 valid AUC，自动保存 best ckpt。


In [25]:
from utils import EarlyStopping
from trainer import PCVRHyFormerRankingTrainer
from torch.utils.tensorboard import SummaryWriter

ckpt_dir = WORK_DIR / 'ckpt'; ckpt_dir.mkdir(exist_ok=True)
tf_dir   = WORK_DIR / 'tb';   tf_dir.mkdir(exist_ok=True)
writer = SummaryWriter(str(tf_dir))

early_stopping = EarlyStopping(
    checkpoint_path=str(ckpt_dir / 'placeholder' / 'model.pt'),
    patience=3, label='model',
)

TRAIN_CFG = dict(
    lr=1e-4,
    num_epochs=2,                       # 演示用，baseline 默认 10
    device=DEVICE,
    save_dir=str(ckpt_dir),
    early_stopping=early_stopping,
    loss_type='bce',
    focal_alpha=0.1, focal_gamma=2.0,
    sparse_lr=0.01,                     # 与 run.sh 一致
    sparse_weight_decay=0.0,
    reinit_sparse_after_epoch=999,      # 演示阶段关闭 MEDA
    reinit_cardinality_threshold=100_000,
    ckpt_params={'layer': MODEL_CFG['num_hyformer_blocks'],
                 'head':  MODEL_CFG['num_heads'],
                 'hidden':MODEL_CFG['d_model']},
    writer=writer,
    schema_path=str(schema_path),
    ns_groups_path=None,
    eval_every_n_steps=0,
    train_config={**MODEL_CFG, **DATA_CFG, 'num_epochs': 2},
)

trainer = PCVRHyFormerRankingTrainer(
    model=model, train_loader=train_loader, valid_loader=valid_loader, **TRAIN_CFG)
print('Trainer 已就绪，开始训练……')
trainer.train()
writer.close()
print('训练完成。最佳 valid AUC =', early_stopping.best_score)
print('最优 ckpt 目录列表:', [p.name for p in ckpt_dir.glob('global_step*')])


04/29/26 16:33:55 - 0:00:01 - Sparse params: 102 tensors, 3,632,448 parameters (Adagrad lr=0.01)
04/29/26 16:33:55 - 0:00:01 - Dense params: 380 tensors, 2,442,241 parameters (AdamW lr=0.0001)
04/29/26 16:33:55 - 0:00:02 - PCVRHyFormerRankingTrainer loss_type=bce, focal_alpha=0.1, focal_gamma=2.0, reinit_sparse_after_epoch=999


Trainer 已就绪，开始训练……
Start training (PCVRHyFormer)


100%|██████████| 16/16 [00:01<00:00,  8.05it/s, loss=0.5451]
04/29/26 16:33:57 - 0:00:04 - Epoch 1, Average Loss: 0.5970252212136984


Start Evaluation (PCVRHyFormer) - validation


100%|██████████| 4/4 [00:00<00:00, 20.95it/s]
04/29/26 16:33:58 - 0:00:04 - Epoch 1 Validation | AUC: 0.5050913998282419, LogLoss: 0.6012571454048157
04/29/26 16:33:58 - 0:00:04 - Removed old best_model dir: /Users/huangjing103/PyCharmMiscProject/taac-codes/playground_workdir/ckpt/global_step16.layer=2.head=4.hidden=64.best_model
04/29/26 16:33:58 - 0:00:04 - Saved checkpoint to /Users/huangjing103/PyCharmMiscProject/taac-codes/playground_workdir/ckpt/global_step16.layer=2.head=4.hidden=64.best_model/model.pt
100%|██████████| 16/16 [00:01<00:00,  8.59it/s, loss=0.4143]
04/29/26 16:34:00 - 0:00:06 - Epoch 2, Average Loss: 0.3951120898127556


Start Evaluation (PCVRHyFormer) - validation


100%|██████████| 4/4 [00:00<00:00, 19.14it/s]
04/29/26 16:34:00 - 0:00:06 - Epoch 2 Validation | AUC: 0.49552202183781124, LogLoss: 0.6318618655204773
04/29/26 16:34:00 - 0:00:06 - model earlyStopping counter: 1 / 3


训练完成。最佳 valid AUC = 0.5050913998282419
最优 ckpt 目录列表: ['global_step16.layer=2.head=4.hidden=64.best_model']


## Section 20 · 评估 + 单 batch 推理

加载最优 ckpt，再调一次 evaluate 拿最终 AUC/LogLoss；
然后演示 `model.predict(inputs)`：返回 `(logits, output_embedding)`，可作向量召回。


In [26]:
# 20.A：用最佳 ckpt 跑一次 valid
import glob as _glob
best_ckpts = sorted(_glob.glob(str(ckpt_dir / '*best*' / 'model.pt')))
if best_ckpts:
    print(f'Loading {best_ckpts[-1]}')
    state = torch.load(best_ckpts[-1], map_location=DEVICE)
    model.load_state_dict(state, strict=False)

# trainer.evaluate() 没有参数，内部直接用 self.valid_loader（也就是上面传进去的 valid_loader）
auc, logloss = trainer.evaluate()
print(f'Final valid AUC = {auc:.4f}, LogLoss = {logloss:.4f}')


Loading /Users/huangjing103/PyCharmMiscProject/taac-codes/playground_workdir/ckpt/global_step16.layer=2.head=4.hidden=64.best_model/model.pt
Start Evaluation (PCVRHyFormer) - validation


100%|██████████| 4/4 [00:00<00:00, 12.49it/s]

Final valid AUC = 0.5051, LogLoss = 0.6013


In [27]:
# 20.B：predict 接口演示（返回 logits + 向量）
model.eval()
with torch.no_grad():
    logits, hidden = model.predict(inputs)
print(f'logits.shape  = {tuple(logits.shape)}')
print(f'hidden.shape  = {tuple(hidden.shape)}  (output embedding，可用作向量召回)')
print()
pcvr = torch.sigmoid(logits[:10]).cpu().numpy().ravel()
uids = inputs.user_int_feats[:10, 0].cpu().numpy()
print('前 10 个用户的 pCVR：')
for u, p in zip(uids, pcvr):
    print(f'  user[0]_id={u:>6d}   pCVR = {p:.4f}')


logits.shape  = (64, 1)
hidden.shape  = (64, 64)  (output embedding，可用作向量召回)

前 10 个用户的 pCVR：
  user[0]_id=    35   pCVR = 0.2289
  user[0]_id=    25   pCVR = 0.2242
  user[0]_id=    82   pCVR = 0.2480
  user[0]_id=   106   pCVR = 0.2366
  user[0]_id=   163   pCVR = 0.3645
  user[0]_id=   166   pCVR = 0.3943
  user[0]_id=    74   pCVR = 0.4256
  user[0]_id=   176   pCVR = 0.4893
  user[0]_id=   132   pCVR = 0.4095
  user[0]_id=   108   pCVR = 0.2294


## 全文走读结束 ✓

到这里你已经：
1. **逐段读完了 baseline `train/model.py` 的全部 16 个类**——RoPE、SwiGLU、RoPEMultiheadAttention（含输出门控 G）、CrossAttention、RankMixerBlock（零参 token mixing）、MultiSeqQueryGenerator、SwiGLU/Transformer/LongerEncoder、HyFormerBlock、Group/RankMixer NS Tokenizer、PCVRHyFormer 主模型；
2. **完整端到端跑通了一次训练**：合成数据 → schema → DataLoader → 模型 forward → BCE loss → 双优化器（Adagrad + AdamW）→ valid AUC + LogLoss；
3. **理解了 baseline 的解耦设计**：trainer 仅依赖 `ModelInput / forward / get_sparse_params / get_dense_params / reinit_high_cardinality_params`，所以本 notebook 自定义的 PCVRHyFormer 也能被原版 trainer 直接接管。

## 推荐下一步实验
- 把 `seq_encoder_type` 切到 `'longer'`，对比 `seq_top_k` 在长序列下的影响。
- 把 `ns_tokenizer_type` 切回 `'group'`，观察 `num_ns` 自由度变化对 `T = num_queries*S + num_ns` 的约束。
- 增大 `num_hyformer_blocks`，看 LongerEncoder 在多 block 串联时「先压到 top_k 再 self-attn」的行为。
- 打开 `reinit_sparse_after_epoch=N`（MEDA），观察高基数 Embedding 的周期性重置如何缓解过拟合。
- 真实数据：`git lfs pull` 拉到完整 `demo_1000.parquet`（甚至 `demo_10w.parquet`），重新跑本 notebook（数据准备 cell 会自动检测并切换）。
